Analytical Model

Architecture DOR:

- p - physical channel (0 - wide bus; 1 - narrow bus)
- v - virtual channel (0 - read/write request, YX routing; 1 - read/write response, XY routing)
- s - source tile
- d - destination tile
- r - routing tile
- i - input direction
- o - output direction

Routing Tensor (RT):
RT[s][d][p][v][r][i][o]

Packet injection rate (f):
f[s][d][p][v] - packet injection rate for route s -> d via physical channel p, virtual channel v:
- f[s][d][0][0] - write request (s - perimeter; d - internal)
- f[s][d][0][1] - read response (s - internal; d - perimeter)
- f[s][d][1][0] - read request (s - perimeter; d - internal)
- f[s][d][1][1] - write response (s - internal; d - perimeter)

Packet Length (L):
L[p][v]

Router packet Arrival rate (A):
A[p][v][r][i][o] = Sum{any s; any d}(f[s][d][p][v] * RT[s][d][p][v][r][i][o])

Input buffer Arrival rate (IA):
IA[p][v][r][i] = Sum{any o}(A[p][v][r][i][o])

Output buffer Arrival rate (OA):
OA[p][v][r][o] = Sum{any i}(A[p][v][r][i][o])

Waiting Time of body flits from other directions in Input buffer (Tbody):
Tbody[p][v][r][i][o] = Sum{any k != i}(A[p][v][r][k][o] / L[p][v] * Sum{q from 1 to L[p][v]}(L - q)) = (L[p][v] - 1) / 2 * Sum{any k != i}(A[p][v][r][k][o])



- [0][0] - write request for horizontal (YX)
- [0][1] - read response for horizontal (XY)
- [0][2] - write request for vertical (XY)
- [0][3] - read response for vertical (YX)
- [1][0] - read request for horizontal (YX)
- [1][1] - write response for horizontal (XY)
- [1][2] - read request for vertical (XY)
- [1][3] - write response for vertical (YX)

- [0][0] -> [1][1]
- [0][2] -> [1][3]
- [1][0] -> [0][1]
- [1][2] -> [0][3]

In [1]:
import numpy as np

## Default params (DOR 2 VC)

In [2]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 2

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM

PACKET_LEN = 1

BUF_DEPTH = 2

In [3]:
class Coord:
    def __init__(self, id=0):
        self.x = id % X_NUM
        self.y = id // X_NUM

    def __repr__(self):
        return f"Coordinate(x={self.x}, y={self.y})"

    def is_perimeter(self):
        return (self.x == 0) or (self.x == X_NUM - 1) or (self.y == 0) or (self.y == Y_NUM - 1)

    def is_vertical(self):
        return (self.x == 0) or (self.x == X_NUM - 1)

    def is_horizontal(self):
        return self.is_perimeter() and not self.is_vertical()

    def id(self):
        return self.y * X_NUM + self.x

In [4]:
# Returns next tile for given current and destination ones
# Algorith DOR
def next_tile_gen_yx(current_tile, dst_tile):
    next = Coord(current_tile.id())
    if current_tile.y < dst_tile.y:
        next.y += 1
    elif current_tile.y > dst_tile.y:
        next.y -= 1
    elif current_tile.x < dst_tile.x:
        next.x += 1
    elif current_tile.x > dst_tile.x:
        next.x -= 1
    return next

def next_tile_gen_xy(current_tile, dst_tile):
    next = Coord(current_tile.id())
    if current_tile.x < dst_tile.x:
        next.x += 1
    elif current_tile.x > dst_tile.x:
        next.x -= 1
    elif current_tile.y < dst_tile.y:
        next.y += 1
    elif current_tile.y > dst_tile.y:
        next.y -= 1
    return next

In [5]:
# Returns directions current_tile output ID and next_tile input ID
def encode_dirs(current_tile, next_tile):
    if current_tile.y < next_tile.y:
        return (SOUTH_IDX, NORTH_IDX)
    elif current_tile.y > next_tile.y:
        return (NORTH_IDX, SOUTH_IDX)
    elif current_tile.x < next_tile.x:
        return (EAST_IDX, WEST_IDX)
    elif current_tile.x > next_tile.x:
        return (WEST_IDX, EAST_IDX)
    return (LOCAL_IDX, LOCAL_IDX)

In [6]:
def calc_collision_possibility_internal(request_possibility, i, o, c, num):
    # The four other indices besides i
    others = [x for x in range(num) if x != i]
    others_num = num - 1
    others_comb_num = 2**others_num
    total = 0.0
    # Loop over all 16 combinations of (k1, k2, k3, k4) in {0,1}^4
    for mask in range(others_comb_num):  # from 0 to 15
        # Count how many bits are 1
        # and build the product F[...]^(k_t) * (1-F[...])^(1-k_t)
        ksum = 0
        prob_product = 1.0
        for bit_idx in range(others_num):
            # k_t is either 0 or 1
            k_t = (mask >> bit_idx) & 1
            p = request_possibility[others[bit_idx]][o]
            if k_t == 1:
                prob_product *= p
                ksum += 1
            else:
                prob_product *= (1 - p)

        # If k_1 + k_2 + k_3 + k_4 = c, add to sum
        if ksum == c:
            total += prob_product
    return total


def calc_collision_possibility(request_possibility, i, o, num):
    '''
    `request_possibilty[i][o]`

    -> `collision_possibility`
    '''
    collision_possibility = 0.0
    for c in range(1, num):
        collision_possibility_internal = calc_collision_possibility_internal(
            request_possibility, i, o, c, num)
        collision_possibility += collision_possibility_internal * \
            (c / (c + 1))
    return collision_possibility


def solve_request_possibility(payload_tensor, flow_control_possibility, max_iter=1000, tol=1e-8):
    '''
    `payload_tensor[i][o]`
    `flow_control_possibility[i][o]`

    -> `request_possibility[i][o]`
    '''
    request_possibility = payload_tensor.copy()

    (input_num, output_num) = payload_tensor.shape

    for _ in range(max_iter):
        request_possibility_old = request_possibility.copy()

        for i in range(input_num):
            for o in range(output_num):
                collision_possibility = calc_collision_possibility(
                    request_possibility, i, o, input_num)
                blocking_possibility = 1.0 - (1.0 - collision_possibility) * (1.0 - flow_control_possibility[i][o])

                denom = 1.0 - blocking_possibility
                if abs(denom) < 1e-14:
                    # Avoid dividing by zero.
                    # Could set F[i][j] to some fallback value.
                    request_possibility[i][o] = 0.999999 if denom < 0 else 0.0
                else:
                    request_possibility[i][o] = payload_tensor[i][o] / denom
                    if request_possibility[i][o] >= 1.0:
                        request_possibility[i][o] = 0.999999

        # Check for convergence
        diff = np.linalg.norm(request_possibility - request_possibility_old)
        if diff < tol:
            break

    # print(f"Finished in {it+1} iterations with diff={diff}")
    return request_possibility


def calc_blocking_possibility(request_possibility, flow_control_possibility):
    '''
        `request_possibility[i][o]`
        `flow_control_possibility[i][o]`

        -> `blocking_possibility[i][o]`
    '''
    blocking_possibility = request_possibility.copy()
    (input_num, output_num) = blocking_possibility.shape

    for i in range(input_num):
        for o in range(output_num):
            collision_possibility = calc_collision_possibility(request_possibility, i, o, input_num)
            blocking_possibility[i][o] = 1.0 - (1.0 - collision_possibility) * (1.0 - flow_control_possibility[i][o])

    return blocking_possibility

In [7]:
def is_traffic_exist(s, d, is_forward, is_vertical_master, s_exclude=list()):
    s_coord = Coord(s)
    d_coord = Coord(d)
    if ROUTING == "DOR":
        if is_forward and (s_coord.is_perimeter() and not d_coord.is_perimeter()):
            return s not in s_exclude
        if not is_forward and (not s_coord.is_perimeter() and d_coord.is_perimeter()):
            return d not in s_exclude
        return False
    else:
        if is_forward and not is_vertical_master and (s_coord.is_horizontal() and not d_coord.is_perimeter()):
            return s not in s_exclude
        if is_forward and is_vertical_master and (s_coord.is_vertical() and not d_coord.is_perimeter()):
            return s not in s_exclude
        if not is_forward and not is_vertical_master and (not s_coord.is_perimeter() and d_coord.is_horizontal()):
            return d not in s_exclude
        if not is_forward and is_vertical_master and (not s_coord.is_perimeter() and d_coord.is_vertical()):
            return d not in s_exclude
        return False

def next_tile_gen_routing(current_tile, d_coord, is_forward, is_vertical_master):
    if ROUTING == "DOR":
        if is_forward:
            return next_tile_gen_yx(current_tile, d_coord)
        else:
            return next_tile_gen_xy(current_tile, d_coord)
    else:
        if is_forward and is_vertical_master:
            return next_tile_gen_xy(current_tile, d_coord)
        elif is_forward and not is_vertical_master:
            return next_tile_gen_yx(current_tile, d_coord)
        elif not is_forward and is_vertical_master:
            return next_tile_gen_yx(current_tile, d_coord)
        else:
            return next_tile_gen_xy(current_tile, d_coord)

def src_dst_routing_tensor(s, d, is_forward, is_vertical_master, s_exclude=list()):
    '''
    -> `routing_tensor[r][i][o]`
    '''
    #              [r]       [i]      [o]
    rt = np.zeros((TILE_NUM, DIR_NUM, DIR_NUM))

    if not is_traffic_exist(s, d, is_forward, is_vertical_master, s_exclude=s_exclude):
        return rt

    s_coord = Coord(s)
    d_coord = Coord(d)
    current_tile = s_coord
    i_cur_dir = LOCAL_IDX
    while current_tile.id() != d:
        next_tile = next_tile_gen_routing(current_tile, d_coord, is_forward, is_vertical_master)
        (o_cur_dir, i_nxt_dir) = encode_dirs(current_tile, next_tile)
        rt[current_tile.id()][i_cur_dir][o_cur_dir] = 1.0
        current_tile = Coord(next_tile.id())
        i_cur_dir = i_nxt_dir
    rt[d][i_cur_dir][LOCAL_IDX] = 1.0

    return rt

def calc_routing_tensor(s_exclude=list()):
    '''
    -> `routing_tensor[p][v][s][d][r][i][o]`
    '''
    routing_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM, TILE_NUM, DIR_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for s in range(TILE_NUM):
                for d in range(TILE_NUM):
                    is_forward = (v % 2 == 0)
                    is_vertical_master = (v > 1)
                    routing_tensor[p][v][s][d] = src_dst_routing_tensor(s, d, is_forward, is_vertical_master, s_exclude=s_exclude)
    return routing_tensor

def calc_packet_length_tensor():
    '''
    -> `packet_length_tensor[p][v]`
    '''
    packet_length_tensor = np.zeros((PHYS_NUM, VIRT_NUM))
    for v in range(VIRT_NUM):
        packet_length_tensor[0][v] = PACKET_LEN * 1.
        packet_length_tensor[1][v] = 1.
    return packet_length_tensor

In [8]:
import csv
import re

def read_traffic_from_csv_file(filename, factor, Y_NUM=10):
    data = list()
    data.append(list())
    data.append(list())
    data.append(list())
    data.append(list())
    data.append(list())
    data.append(list())
    result_names = [0, Y_NUM+1, 2 *
                (Y_NUM+1), 3*(Y_NUM+1), 4*(Y_NUM+1), 5*(Y_NUM+1)]

    row_idx = 0
    data_idx = -1

    with open(filename, newline='') as csvfile:
        spamreader = csv.reader(
            csvfile, delimiter=';', quotechar=';')
        for row in spamreader:
            if row_idx not in result_names:
                data[data_idx].append(list(map(int, row[:-1])))
            else:
                data_idx += 1
            row_idx += 1

    traffic = np.zeros((PHYS_NUM, TILE_NUM, TILE_NUM))

    mem_array_size = (X_NUM - 2) * (Y_NUM - 2)
    for s in range(TILE_NUM):
        s_coord = Coord(s)
        if s_coord.is_perimeter():
            for d in range(TILE_NUM):
                d_coord = Coord(d)
                if not d_coord.is_perimeter():
                    traffic[0][s][d] = factor * data[2][s_coord.x][s_coord.y] / 100000.0 / mem_array_size
                    traffic[1][s][d] = factor * data[3][s_coord.x][s_coord.y] / 100000.0 / mem_array_size
    return traffic

def read_traffic_from_file(filename, Y_NUM=10):
    """
    Reads a file with lines of the form:
        [x][0][0] -> [5][8]: 87
    or
        [y][1][0] -> [6][5]: 66

    and returns a nested dictionary of counts:
        result[channel][source_id][dest_id] = <count>

    :param filename: file containing the lines to parse
    :param X_NUM: used in the ID formula (X * X_NUM + Y)
    :return: 
    """
    # Regex to capture:
    #   group(1) = channel (x or y)
    #   group(2) = source X
    #   group(3) = source Y
    #   group(4) = dest X
    #   group(5) = dest Y
    pattern = re.compile(
        r'\[([xy])\]\[(\d+)\]\[(\d+)\]\s*->\s*\[(\d+)\]\[(\d+)\]:'
    )

    # Default dict structure: {channel: {sourceID: {destID: count}}}
    counts = np.zeros((PHYS_NUM, TILE_NUM, TILE_NUM))

    with open(filename, 'r') as f:
        for line in f:
            match = pattern.search(line.strip())
            if match:
                channel = match.group(1)
                sX = int(match.group(2))
                sY = int(match.group(3))
                dX = int(match.group(4))
                dY = int(match.group(5))

                # Compute source and destination IDs
                s_id = sX + sY * Y_NUM
                d_id = dX + dY * Y_NUM

                # Increment count
                if channel == 'x':
                    counts[0][s_id][d_id] += 1.0
                else:
                    counts[1][s_id][d_id] += 1.0

    return counts / 100000.0

def calc_packet_injection_rate_tensor(pir, s_exclude = list()):
    '''
    `pir`

    -> `packet_injection_rate_tensor[p][v][s][d]`
    '''
    packet_injection_rate_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for s in range(TILE_NUM):
                for d in range(TILE_NUM):
                    is_forward = (v % 2 == 0)
                    is_vertical_master = (v > 1)
                    if is_traffic_exist(s, d, is_forward, is_vertical_master, s_exclude=s_exclude):
                        # print(f"traffic exist [{p}][{v}][{s}][{d}]")
                        packet_injection_rate_tensor[p][v][s][d] = pir / 2 if ROUTING == "DOR" and VIRT_NUM == 4 else pir


    return packet_injection_rate_tensor

def read_packet_injection_rate_tensor_from_file(pir, factor):
    '''
    -> `packet_injection_rate_tensor[p][v][s][d]`
    '''
    packet_injection_rate_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM))
    file_name = f"../sim/results/latencies/{ROUTING.lower()}_mesh_10x10_{ROUTING}_mp_1_pir_{pir}_ps_1_bs_1.csv"
    traffic = read_traffic_from_csv_file(file_name, factor)
    for _ in range(PHYS_NUM):
        for s in range(TILE_NUM):
            for d in range(TILE_NUM):
                s_coord = Coord(s)
                if s_coord.is_perimeter():
                    if ROUTING == "DOR":
                        packet_injection_rate_tensor[0][0][s][d] = traffic[0][s][d]
                        packet_injection_rate_tensor[1][1][d][s] = traffic[0][s][d]
                        packet_injection_rate_tensor[1][0][s][d] = traffic[1][s][d]
                        packet_injection_rate_tensor[0][1][d][s] = traffic[1][s][d]
                    else:
                        if s_coord.is_vertical():
                            packet_injection_rate_tensor[0][2][s][d] = traffic[0][s][d]
                            packet_injection_rate_tensor[1][3][d][s] = traffic[0][s][d]
                            packet_injection_rate_tensor[1][2][s][d] = traffic[1][s][d]
                            packet_injection_rate_tensor[0][3][d][s] = traffic[1][s][d]
                        else:
                            packet_injection_rate_tensor[0][0][s][d] = traffic[0][s][d]
                            packet_injection_rate_tensor[1][1][d][s] = traffic[0][s][d]
                            packet_injection_rate_tensor[1][0][s][d] = traffic[1][s][d]
                            packet_injection_rate_tensor[0][1][d][s] = traffic[1][s][d]
    return packet_injection_rate_tensor

In [9]:
def calc_router_packet_payload_tensor(routing_tensor, packet_injection_rate_tensor, packet_length_tensor):
    '''
    `routing_tensor[p][v][s][d][r][i][o]`
    `packet_injection_rate_tensor[p][v][s][d]`
    `packet_length_tensor[p][v]`

    -> `router_packet_payload_tensor[p][v][r][i][o]`
    '''
    multiplied = routing_tensor * packet_injection_rate_tensor[..., np.newaxis, np.newaxis, np.newaxis]
    router_packet_payload_tensor = multiplied.sum(axis=(2, 3))
    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for i in range(DIR_NUM):
                    for o in range(DIR_NUM):
                        if router_packet_payload_tensor[p][v][r][i][o] * packet_length_tensor[p][v] >= 1.0:
                            router_packet_payload_tensor[p][v][r][i][o] = 0.999999 / packet_length_tensor[p][v]
    # [p] [v] [r] [i] [o]
    return router_packet_payload_tensor

def calc_router_flit_payload_tensor(router_packet_payload_tensor, packet_length_tensor):
    '''
    `router_packet_payload_tensor[p][v][r][i][o]`
    `packet_length_tensor[p][v]`

    -> `router_flit_payload_tensor[p][v][r][i][o]`
    '''
    router_flit_payload_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM, DIR_NUM))
    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for i in range(DIR_NUM):
                    for o in range(DIR_NUM):
                        router_flit_payload_tensor[p][v][r][i][o] = router_packet_payload_tensor[p][v][r][i][o] * packet_length_tensor[p][v]
    return router_flit_payload_tensor

In [10]:
def calc_router_input_packet_payload_tensor(router_packet_payload_tensor):
    '''
    `router_packet_payload_tensor[p][v][r][i][o]`

    -> `router_input_packet_payload_tensor[p][v][r][i]`
    '''
    # [p] [v] [r] [i]
    return router_packet_payload_tensor.sum(axis=-1)

In [11]:
def calc_router_output_packet_payload_tensor(router_packet_payload_tensor):
    '''
    `router_packet_payload_tensor[p][v][r][i][o]`

    -> `router_output_packet_payload_tensor[p][v][r][o]`
    '''
    # [p] [v] [r] [o]
    return router_packet_payload_tensor.sum(axis=3)

In [12]:
def calc_input_blocking_possibility_tensor(router_packet_payload_tensor, router_input_flow_control_possibility_tensor):
    '''
    `router_packet_payload_tensor[p][v][r][i][o]`
    `router_input_flow_control_possibility_tensor[p][v][r][o]`

    -> `input_blocking_possibility_tensor[p][v][r][i][o]`
    '''
    #                                             [p]       [v]       [r]       [i]      [o]
    input_blocking_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                request_possibility_solution = solve_request_possibility(
                    router_packet_payload_tensor[p][v][r], router_input_flow_control_possibility_tensor[p][v][r], max_iter=500, tol=1e-10)
                input_blocking_possibility_tensor[p][v][r] = calc_blocking_possibility(
                    request_possibility_solution, router_input_flow_control_possibility_tensor[p][v][r])

    return input_blocking_possibility_tensor

def calc_output_blocking_possibility_tensor(router_flit_payload_tensor, router_output_flow_control_possibility_tensor):
    '''
    `router_flit_payload_tensor[p][v][r][i][o]`
    `router_output_flow_control_possibility_tensor[p][v][r][o]`

    -> `output_blocking_possibility_tensor[p][v][r][o]`
    '''
    output_blocking_possibility_tensor_reshape = np.zeros((PHYS_NUM, TILE_NUM, VIRT_NUM, DIR_NUM))
    # [p][v][r][o]
    router_output_packet_payload_tensor = calc_router_output_packet_payload_tensor(router_flit_payload_tensor)
    # [p][r][v][o]
    router_output_packet_payload_tensor_reshape = np.zeros((PHYS_NUM, TILE_NUM, VIRT_NUM, DIR_NUM))
    router_output_flow_control_possibility_tensor_reshape = np.zeros((PHYS_NUM, TILE_NUM, VIRT_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for o in range(DIR_NUM):
                    router_output_packet_payload_tensor_reshape[p][r][v][o] = router_output_packet_payload_tensor[p][v][r][o]
                    router_output_flow_control_possibility_tensor_reshape[p][r][v][o] = router_output_flow_control_possibility_tensor[p][v][r][o]
    for p in range(PHYS_NUM):
        for r in range(TILE_NUM):
            request_possibility_solution = solve_request_possibility(
                router_output_packet_payload_tensor_reshape[p][r], router_output_flow_control_possibility_tensor_reshape[p][r], max_iter=500, tol=1e-10)
            output_blocking_possibility_tensor_reshape[p][r] = calc_blocking_possibility(
                request_possibility_solution, router_output_flow_control_possibility_tensor_reshape[p][r])
    output_blocking_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for o in range(DIR_NUM):
                    output_blocking_possibility_tensor[p][v][r][o] = output_blocking_possibility_tensor_reshape[p][r][v][o]
    return output_blocking_possibility_tensor

In [13]:
def calc_input_blocking_time_tensor(input_blocking_possibility_tensor, packet_length_tensor):
    '''
    `input_blocking_possibility_tensor[p][v][r][i][o]`

    -> `input_blocking_time_tensor[p][v][r][i][o]`
    '''
    # input_blocking_time_tensor = input_blocking_possibility_tensor.copy()

    # for p in range(PHYS_NUM):
    #     for v in range(VIRT_NUM):
    #         for r in range(TILE_NUM):
    #             for i in range(DIR_NUM):
    #                 for o in range(DIR_NUM):
    #                     if abs(1 - input_blocking_time_tensor[p][v][r][i][o]) < 1e-9:
    #                         input_blocking_time_tensor[p][v][r][i][o] = 0.999999
    #                     input_blocking_time_tensor[p][v][r][i][o] /= (1 - input_blocking_time_tensor[p][v][r][i][o])
    #                     input_blocking_time_tensor[p][v][r][i][o] *= packet_length_tensor[p][v]

    # return input_blocking_time_tensor

    input_blocking_time_tensor = input_blocking_possibility_tensor.copy()  # сохранить семантику оригинала

    # защита x≈1
    mask = np.abs(1.0 - input_blocking_time_tensor) < 1e-9
    if mask.any():
        input_blocking_time_tensor[mask] = 0.999999

    # y = x / (1 - x)
    with np.errstate(divide='ignore', invalid='ignore'):
        np.divide(input_blocking_time_tensor, (1.0 - input_blocking_time_tensor), out=input_blocking_time_tensor)

    # умножаем на L[p,v]
    input_blocking_time_tensor *= packet_length_tensor[..., None, None, None]
    return input_blocking_time_tensor

def calc_input_body_block_tensor(router_packet_payload_tensor, packet_length_tensor):
    '''
    `router_packet_payload_tensor[p][v][r][i][o]`
    `packet_length_tensor[p][v]`

    -> `input_body_block_tensor[p][v][r][i][o]`
    '''
    # input_body_block_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM, DIR_NUM))
    # for p in range(PHYS_NUM):
    #     for v in range(VIRT_NUM):
    #         for r in range(TILE_NUM):
    #             for i in range(DIR_NUM):
    #                 for k in range(DIR_NUM):
    #                     for o in range(DIR_NUM):
    #                         if k != i:
    #                             input_body_block_tensor[p][v][r][i][o] += router_packet_payload_tensor[p][v][r][k][o]
    #         input_body_block_tensor[p][v] *= packet_length_tensor[p][v]
    #         input_body_block_tensor[p][v] *= (packet_length_tensor[p][v] - 1.) / 2.
    # return input_body_block_tensor
    # Сумма по всем входам k (ось I=3), keepdims=True -> (P,V,R,1,O)
    sum_all = router_packet_payload_tensor.sum(axis=3, keepdims=True)                    # (P,V,R,1,O)

    # Для каждого i: sum_{k != i} X[k] = (sum_k X[k]) - X[i]
    body = sum_all - router_packet_payload_tensor                                        # (P,V,R,I,O)

    # Умножаем на L * (L-1)/2, расшаривая по R,I,O
    factor = packet_length_tensor * (packet_length_tensor - 1.0) / 2.0                              # (P,V)
    body *= factor[..., None, None, None]                     # broadcast

    return body

def calc_output_blocking_time_tensor(output_blocking_possibility_tensor):
    '''
    `output_blocking_possibility_tensor[p][v][r][o]`

    -> `output_blocking_time_tensor[p][v][r][o]`
    '''
    # output_blocking_time_tensor = output_blocking_possibility_tensor.copy()

    # for p in range(PHYS_NUM):
    #     for v in range(VIRT_NUM):
    #         for r in range(TILE_NUM):
    #             for o in range(DIR_NUM):
    #                 if abs(1 - output_blocking_time_tensor[p][v][r][o]) < 1e-9:
    #                     output_blocking_time_tensor[p][v][r][o] = 0.999999
    #                 output_blocking_time_tensor[p][v][r][o] /= (1 - output_blocking_time_tensor[p][v][r][o])
    # return output_blocking_time_tensor
    output_blocking_time_tensor = output_blocking_possibility_tensor.copy()  # сохранить семантику .copy()

    # Замена значений, слишком близких к 1
    mask = np.abs(1.0 - output_blocking_time_tensor) < 1e-9
    if mask.any():
        output_blocking_time_tensor[mask] = 0.999999

    # Векторизированное деление
    output_blocking_time_tensor /= (1.0 - output_blocking_time_tensor)
    return output_blocking_time_tensor

In [14]:
def calc_input_request_handle_time_tensor(input_blocking_time_tensor, input_body_block_tensor, packet_length_tensor):
    '''
    `input_blocking_time_tensor[p][v][r][i][o]`
    `input_body_block_tensor[p][v][r][i][o]`
    `packet_length_tensor[p][v]`

    -> `input_request_handle_time_tensor[p][v][r][i][o]`
    '''
    # input_request_handle_time_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM, DIR_NUM))
    # for p in range(PHYS_NUM):
    #     for v in range(VIRT_NUM):
    #         for r in range(TILE_NUM):
    #             for i in range(DIR_NUM):
    #                 for o in range(DIR_NUM):
    #                     input_request_handle_time_tensor[p][v][r][i][o] = input_blocking_time_tensor[p][v][r][i][o] + input_body_block_tensor[p][v][r][i][o] + packet_length_tensor[p][v]
    # return input_request_handle_time_tensor
    return (input_blocking_time_tensor
            + input_body_block_tensor
            + packet_length_tensor[..., None, None, None])

def calc_output_request_handle_time_tensor(output_blocking_time_tensor):
    '''
    `output_blocking_time_tensor[p][v][r][o]`

    -> `output_request_handle_time_tensor[p][v][r][o]`
    '''
    return output_blocking_time_tensor.copy() + 1.

In [15]:
def calc_mean_input_handle_time_tensor(router_packet_payload_tensor, request_handle_time_tensor):
    '''
    `router_packet_payload_tensor[p][v][r][i][o]`
    `request_handle_time_tensor[p][v][r][i][o]`

    -> `mean_input_handle_time_tensor[p][v][r][i]`
    '''
    mean_input_handle_time_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
    router_input_packet_payload_tensor = calc_router_input_packet_payload_tensor(router_packet_payload_tensor)

    handle_time_tensor = router_packet_payload_tensor * request_handle_time_tensor
    sum_input_handle_time_tensor = handle_time_tensor.sum(axis=-1)

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for i in range(DIR_NUM):
                    mean_input_handle_time_tensor[p][v][r][i] = 0. if (abs(sum_input_handle_time_tensor[p][v][r][i]) < 1e-9) else sum_input_handle_time_tensor[p][v][r][i] / \
                        router_input_packet_payload_tensor[p][v][r][i]

    return mean_input_handle_time_tensor

In [16]:
def mm1n_queue(lambda_arrival, E_T, N):
    '''
    `lambda_arrival` -- intensivity of requests
    `E_T` -- average request handle time
    `N` -- number of queue entries

    -> (mean_depth, p[N]) -- mean queue depth and probability of N requests in the queue
    '''
    utilization = lambda_arrival * E_T
    pk_values = []
    if abs(utilization - 1.0) < 1e-9:
        pk_values = [0. for _ in range(N+1)]
        pk_values[N] = 0.999999
    else:
        pk_values = [(1 - utilization) * utilization**k / (1 - utilization**(N+1))
                 for k in range(N+1)]
    mean_depth = sum((k - 1) * pk_values[k] for k in range(1, N+1))
    return (mean_depth, pk_values[N])

In [17]:
def calc_mean_input_queue_depth_tensor(router_input_packet_payload_tensor, mean_input_handle_time_tensor):
    '''
    `router_input_packet_payload_tensor[p][v][r][i]`
    `mean_input_handle_time_tensor[p][v][r][i]`

    -> `(mean_input_queue_depth_tensor[p][v][r][i], input_queue_full_probability_tensor[p][v][r][i])`
    '''
    mean_input_queue_depth_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
    input_queue_full_probability_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for i in range(DIR_NUM):
                    (mean_input_queue_depth_tensor[p][v][r][i], input_queue_full_probability_tensor[p][v][r][i]) = mm1n_queue(
                        router_input_packet_payload_tensor[p][v][r][i], mean_input_handle_time_tensor[p][v][r][i], BUF_DEPTH)
    return (mean_input_queue_depth_tensor, input_queue_full_probability_tensor)

def calc_mean_output_queue_depth_tensor(router_output_flit_payload_tensor, mean_output_handle_time_tensor):
    '''
    `router_output_flit_payload_tensor[p][v][r][o]`
    `mean_output_handle_time_tensor[p][v][r][o]`

    -> `(mean_output_queue_depth_tensor[p][v][r][o], output_queue_full_probability_tensor[p][v][r][o])`
    '''
    mean_output_queue_depth_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
    output_queue_full_probability_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                for o in range(DIR_NUM):
                    (mean_output_queue_depth_tensor[p][v][r][o], output_queue_full_probability_tensor[p][v][r][o]) = mm1n_queue(
                        router_output_flit_payload_tensor[p][v][r][o], mean_output_handle_time_tensor[p][v][r][o], BUF_DEPTH)
    return (mean_output_queue_depth_tensor, output_queue_full_probability_tensor)

In [18]:
def calc_input_average_latency(routing_tensor, mean_input_queue_depth_tensor, mean_input_handle_time_tensor, packet_length):
    '''
    - `routing_tensor[r][i][o]`
    - `mean_input_queue_depth_tensor[r][i]`
    - `mean_input_handle_time_tensor[r][i]`

    -> `input_average_latency`
    '''
    input_routing_tensor = routing_tensor.sum(axis=-1)

    input_average_latency = 0.0
    for r in range(TILE_NUM):
        for i in range(DIR_NUM):
            if abs(input_routing_tensor[r][i]) > 1e-9:
                input_average_latency += (mean_input_queue_depth_tensor[r][i] + 1.) * \
                    mean_input_handle_time_tensor[r][i] - packet_length + 1.
    return input_average_latency

def calc_output_average_latency(routing_tensor, mean_output_queue_depth_tensor, mean_output_handle_time_tensor):
    '''
    - `routing_tensor[r][i][o]`
    - `mean_output_queue_depth_tensor[r][o]`
    - `mean_output_handle_time_tensor[r][o]`

    -> `output_average_latency`
    '''
    output_routing_tensor = routing_tensor.sum(axis=1)

    output_average_latency = 0.0
    for r in range(TILE_NUM):
        for o in range(DIR_NUM):
            if abs(output_routing_tensor[r][o]) > 1e-9:
                output_average_latency += (mean_output_queue_depth_tensor[r][o] + 1.) * \
                    mean_output_handle_time_tensor[r][o]
    return output_average_latency

In [19]:
def calc_average_latency_tensor(routing_tensor, mean_input_queue_depth_tensor, mean_input_handle_time_tensor, mean_output_queue_depth_tensor, mean_output_handle_time_tensor, packet_length_tensor):
    '''
    - `routing_tensor[p][v][s][d][r][i][o]`
    - `mean_input_queue_depth_tensor[p][v][r][i]`
    - `mean_input_handle_time_tensor[p][v][r][i]`
    - `mean_output_queue_depth_tensor[p][v][r][o]`
    - `mean_output_handle_time_tensor[p][v][r][o]`
    - `packet_length_tensor[p][v]`

    -> `average_latency_tensor[p][v][s][d]`
    '''
    average_latency_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM))
    input_average_latency_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM))
    output_average_latency_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for s in range(TILE_NUM):
                for d in range(TILE_NUM):
                    if abs(routing_tensor[p][v][s][d].sum()) > 1e-9:
                        input_average_latency = calc_input_average_latency(
                            routing_tensor[p][v][s][d], mean_input_queue_depth_tensor[p][v], mean_input_handle_time_tensor[p][v], packet_length_tensor[p][v])
                        output_average_latency = calc_output_average_latency(
                            routing_tensor[p][v][s][d], mean_output_queue_depth_tensor[p][v], mean_output_handle_time_tensor[p][v])
                        average_latency_tensor[p][v][s][d] = input_average_latency + output_average_latency + packet_length_tensor[p][v] - 1.
                        input_average_latency_tensor[p][v][s][d] = input_average_latency
                        output_average_latency_tensor[p][v][s][d] = output_average_latency

                # print(Coord(s), Coord(d), average_latency[s][d])

    return (average_latency_tensor, input_average_latency_tensor, output_average_latency_tensor)

In [20]:
def update_input_flow_control_possibility_tensor(output_queue_full_probability_tensor):
    '''
    `output_queue_full_probability_tensor[p][v][r][o]`

    -> `router_input_flow_control_possibility_tensor[p][v][r][i][o]`
    '''
    expanded = np.expand_dims(output_queue_full_probability_tensor, axis=3)
    return np.repeat(expanded, DIR_NUM, axis=3)

def update_output_flow_control_possibility_tensor(input_queue_full_probability_tensor):
    '''
    input_queue_full_probability_tensor[p][v][r][i]

    -> router_output_flow_control_possibility_tensor[p][v][r][o]
    '''
    output_flow_control_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for r in range(TILE_NUM):
                r_coord = Coord(r)
                if r_coord.y > 0:
                    r_coord_north = Coord(r)
                    r_coord_north.y -= 1
                    output_flow_control_possibility_tensor[p][v][r][NORTH_IDX] = input_queue_full_probability_tensor[p][v][r_coord_north.id()][SOUTH_IDX]
                if r_coord.y < Y_NUM - 1:
                    r_coord_south = Coord(r)
                    r_coord_south.y += 1
                    output_flow_control_possibility_tensor[p][v][r][SOUTH_IDX] = input_queue_full_probability_tensor[p][v][r_coord_south.id()][NORTH_IDX]
                if r_coord.x > 0:
                    r_coord_west = Coord(r)
                    r_coord_west.x -= 1
                    output_flow_control_possibility_tensor[p][v][r][WEST_IDX] = input_queue_full_probability_tensor[p][v][r_coord_west.id()][EAST_IDX]
                if r_coord.x < X_NUM - 1:
                    r_coord_east = Coord(r)
                    r_coord_east.x += 1
                    output_flow_control_possibility_tensor[p][v][r][EAST_IDX] = input_queue_full_probability_tensor[p][v][r_coord_east.id()][WEST_IDX]
                if not r_coord.is_perimeter():
                    p_inp = (p + 1) % PHYS_NUM
                    v_inp = (v + 1) % VIRT_NUM
                    output_flow_control_possibility_tensor[p][v][r][LOCAL_IDX] = input_queue_full_probability_tensor[p_inp][v_inp][r][LOCAL_IDX]
    return output_flow_control_possibility_tensor

In [21]:
def calc_average_latency_pir_pipeline(pir, s_exclude=list(), factor=1, max_iter=10, tol=1e-9):
  routing_tensor = calc_routing_tensor(s_exclude=s_exclude)
  #                                                        [p]       [v]       [r]       [i]      [o]
  router_input_flow_control_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM, DIR_NUM))
  #                                                         [p]       [v]       [r]       [o]
  router_output_flow_control_possibility_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, DIR_NUM))
  # packet_injection_rate_tensor = read_packet_injection_rate_tensor_from_file(pir, factor)
  pir_per_mem_tile = pir / (X_NUM - 2) / (Y_NUM - 2)
  packet_injection_rate_tensor = calc_packet_injection_rate_tensor(pir_per_mem_tile, s_exclude=s_exclude)
  packet_length_tensor = calc_packet_length_tensor()
  router_packet_payload_tensor = calc_router_packet_payload_tensor(routing_tensor, packet_injection_rate_tensor, packet_length_tensor)
  router_flit_payload_tensor = calc_router_flit_payload_tensor(router_packet_payload_tensor, packet_length_tensor)
  router_input_packet_payload_tensor = calc_router_input_packet_payload_tensor(router_packet_payload_tensor)
  router_output_flit_payload_tensor = calc_router_output_packet_payload_tensor(router_flit_payload_tensor)
  for it in range(max_iter):
    input_blocking_possibility_tensor = calc_input_blocking_possibility_tensor(router_packet_payload_tensor, router_input_flow_control_possibility_tensor)
    input_blocking_time_tensor = calc_input_blocking_time_tensor(input_blocking_possibility_tensor, packet_length_tensor)
    input_body_block_tensor = calc_input_body_block_tensor(router_packet_payload_tensor, packet_length_tensor)
    input_request_handle_time_tensor = calc_input_request_handle_time_tensor(input_blocking_time_tensor, input_body_block_tensor, packet_length_tensor)
    mean_input_handle_time_tensor = calc_mean_input_handle_time_tensor(router_packet_payload_tensor, input_request_handle_time_tensor)
    (mean_input_queue_depth_tensor, input_queue_full_probability_tensor) = calc_mean_input_queue_depth_tensor(router_input_packet_payload_tensor, mean_input_handle_time_tensor)
    output_blocking_possibility_tensor = calc_output_blocking_possibility_tensor(router_flit_payload_tensor, router_output_flow_control_possibility_tensor)
    output_blocking_time_tensor = calc_output_blocking_time_tensor(output_blocking_possibility_tensor)
    output_request_handle_time_tensor = calc_output_request_handle_time_tensor(output_blocking_time_tensor)
    mean_output_handle_time_tensor = output_request_handle_time_tensor.copy()
    (mean_output_queue_depth_tensor, output_queue_full_probability_tensor) = calc_mean_output_queue_depth_tensor(router_output_flit_payload_tensor, mean_output_handle_time_tensor)

    router_input_flow_control_possibility_tensor_old = router_input_flow_control_possibility_tensor.copy()
    router_output_flow_control_possibility_tensor_old = router_output_flow_control_possibility_tensor.copy()
    router_input_flow_control_possibility_tensor = update_input_flow_control_possibility_tensor(output_queue_full_probability_tensor)
    router_output_flow_control_possibility_tensor = update_output_flow_control_possibility_tensor(input_queue_full_probability_tensor)
    input_diff = np.linalg.norm(router_input_flow_control_possibility_tensor - router_input_flow_control_possibility_tensor_old)
    output_diff = np.linalg.norm(router_output_flow_control_possibility_tensor - router_output_flow_control_possibility_tensor_old)
    if input_diff < tol and output_diff < tol:
      break
  print(f"finished in iter={it}; input_diff={input_diff}; output_diff={output_diff}")
  return calc_average_latency_tensor(routing_tensor, mean_input_queue_depth_tensor, mean_input_handle_time_tensor, mean_output_queue_depth_tensor, mean_output_handle_time_tensor, packet_length_tensor)

In [22]:
def calculation_task(pir, sender):
  sender.send(calc_average_latency_pir_pipeline(pir, max_iter=50))

# DOR 2 VC (packet_length = 1)

In [23]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 2

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM

PACKET_LEN = 1

BUF_DEPTH = 2

In [24]:
from multiprocessing import Pipe, Process
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450, 0.475, 0.500, 0.525, 0.55, 0.575, 0.6, 0.625, 0.650, 0.675, 0.700, 0.725, 0.750]
# pir_list = [0.025, 0.05]
pir_list = [pir / PACKET_LEN for pir in pir_list]
calculation_tasks = list()
task_result_receivers = list()

for pir in pir_list:
  result_sender, result_receiver = Pipe()
  p = Process(target=calculation_task, args=(pir, result_sender))
  p.start()
  calculation_tasks.append(p)
  task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  input_lat_wreq.append(input_lat[0][0].sum() / PAIR_NUM + 0.5)
  output_lat_wreq.append(output_lat[0][0].sum() / PAIR_NUM + 0.5)
  lat_wreq.append(lat[0][0].sum() / PAIR_NUM + 1)
  input_lat_wresp.append(input_lat[1][1].sum() / PAIR_NUM + 0.5)
  output_lat_wresp.append(output_lat[1][1].sum() / PAIR_NUM + 0.5)
  lat_wresp.append(lat[1][1].sum() / PAIR_NUM + 1)
  input_lat_rreq.append(input_lat[1][0].sum() / PAIR_NUM + 0.5)
  output_lat_rreq.append(output_lat[1][0].sum() / PAIR_NUM + 0.5)
  lat_rreq.append(lat[1][0].sum() / PAIR_NUM + 1)
  input_lat_rresp.append(input_lat[0][1].sum() / PAIR_NUM + 0.5)
  output_lat_rresp.append(output_lat[0][1].sum() / PAIR_NUM + 0.5)
  lat_rresp.append(lat[0][1].sum() / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][0][0][11] + 1
  test_wresp = lat[1][1][11][0] + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=4; input_diff=8.342765655728936e-11; output_diff=3.564178909630351e-11
0.025 done
latency_wreq = 17.821128301628413; latency_wresp = 17.826889281112177; latency_write = 35.64801758274059
latency_rreq = 17.821128301628413; latency_rresp = 17.826889281112177; latency_read = 35.64801758274059
latency_write [0][0] -> [1][1]: wreq = 7.0237536903107305; wresp = 7.013545124333856; write = 14.037298814644586
finished in iter=6; input_diff=3.24072246124511e-11; output_diff=1.3645358846543129e-11
0.05 done
latency_wreq = 18.056747532419266; latency_wresp = 18.068891442473824; latency_write = 36.12563897489309
latency_rreq = 18.056747532419266; latency_rresp = 18.068891442473824; latency_read = 36.12563897489309
latency_write [0][0] -> [1][1]: wreq = 7.066136018785864; wresp = 7.041000913243114; write = 14.107136932028979
finished in iter=7; input_diff=4.406126175846716e-10; output_diff=1.3323414866613564e-10
0.075 done
latency_wreq = 18.38350073496853; latency_wresp = 18.4027441

In [25]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("dor_results_p1.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [26]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.025: wreq 17.821128301628413; wresp 17.826889281112177; w 35.64801758274059; rreq 17.821128301628413; rresp 17.826889281112177; r 35.64801758274059
0.05: wreq 18.056747532419266; wresp 18.068891442473824; w 36.12563897489309; rreq 18.056747532419266; rresp 18.068891442473824; r 36.12563897489309
0.075: wreq 18.38350073496853; wresp 18.402744149094723; w 36.78624488406325; rreq 18.38350073496853; rresp 18.402744149094723; r 36.78624488406325
0.1: wreq 18.822180096399094; wresp 18.84938680410157; w 37.671566900500665; rreq 18.822180096399094; rresp 18.84938680410157; r 37.671566900500665
0.125: wreq 19.41737892694999; wresp 19.453581573863524; w 38.87096050081351; rreq 19.41737892694999; rresp 19.453581573863524; r 38.87096050081351
0.15: wreq 20.295887860753044; wresp 20.341745060849117; w 40.63763292160216; rreq 20.295887860753044; rresp 20.341745060849117; r 40.63763292160216
0.175: wreq 22.43490922947522; wresp 22.389854100411057; w 44.82476332988628; rreq 22.43490922947522; rresp 

# DOR 2 VC (packet_length = 2)

In [23]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 2

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM

PACKET_LEN = 2

BUF_DEPTH = 2

In [26]:
from multiprocessing import Pipe, Process
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400]
# pir_list = [0.075]
pir_list = [pir / PACKET_LEN for pir in pir_list]
calculation_tasks = list()
task_result_receivers = list()

for pir in pir_list:
  result_sender, result_receiver = Pipe()
  p = Process(target=calculation_task, args=(pir, result_sender))
  p.start()
  calculation_tasks.append(p)
  task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  input_lat_wreq.append(input_lat[0][0].sum() / PAIR_NUM + 0.5)
  output_lat_wreq.append(output_lat[0][0].sum() / PAIR_NUM + 0.5)
  lat_wreq.append(lat[0][0].sum() / PAIR_NUM + 1)
  input_lat_wresp.append(input_lat[1][1].sum() / PAIR_NUM + 0.5)
  output_lat_wresp.append(output_lat[1][1].sum() / PAIR_NUM + 0.5)
  lat_wresp.append(lat[1][1].sum() / PAIR_NUM + 1)
  input_lat_rreq.append(input_lat[1][0].sum() / PAIR_NUM + 0.5)
  output_lat_rreq.append(output_lat[1][0].sum() / PAIR_NUM + 0.5)
  lat_rreq.append(lat[1][0].sum() / PAIR_NUM + 1)
  input_lat_rresp.append(input_lat[0][1].sum() / PAIR_NUM + 0.5)
  output_lat_rresp.append(output_lat[0][1].sum() / PAIR_NUM + 0.5)
  lat_rresp.append(lat[0][1].sum() / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][0][0][11] + 1
  test_wresp = lat[1][1][11][0] + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=4; input_diff=5.820880877246203e-11; output_diff=2.4819528477495156e-11
0.0125 done
latency_wreq = 18.87053962989728; latency_wresp = 17.73689947892693; latency_write = 36.60743910882421
latency_rreq = 17.734241817239845; latency_rresp = 18.881890761187567; latency_read = 36.61613257842741
latency_write [0][0] -> [1][1]: wreq = 8.043342014917291; wresp = 7.004993784965455; write = 15.048335799882746
finished in iter=6; input_diff=2.1928258805004024e-11; output_diff=9.177693883004625e-12
0.025 done
latency_wreq = 19.188952676745416; latency_wresp = 17.826889281124313; latency_write = 37.015841957869725
latency_rreq = 17.821729162664447; latency_rresp = 19.212528626418187; latency_read = 37.03425778908263
latency_write [0][0] -> [1][1]: wreq = 8.115532736962525; wresp = 7.013545124333932; write = 15.129077861296457
finished in iter=7; input_diff=2.8794810209338474e-10; output_diff=8.484655922685735e-11
0.0375 done
latency_wreq = 19.63268625700491; latency_wresp = 17.9372

In [27]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("dor_results_p2.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [33]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.0125: wreq 18.87087861815172; wresp 17.73689947892693; w 36.60777809707865; rreq 17.734241829556193; rresp 18.882248030637342; r 36.61648986019354
0.025: wreq 19.19144324915024; wresp 17.826889281124316; w 37.018332530274556; rreq 17.821729417594053; rresp 19.215133244462304; r 37.03686266205636
0.0375: wreq 19.64254350615666; wresp 17.93722384715717; w 37.57976735331383; rreq 17.929723058519997; rresp 19.67970291371956; r 37.609425972239556
0.05: wreq 20.2566511806641; wresp 18.068891442871525; w 38.325542623535625; rreq 18.059221903390227; rresp 20.308715724262633; r 38.367937627652864
0.0625: wreq 21.111433372012034; wresp 18.22336480027737; w 39.3347981722894; rreq 18.21171678530684; rresp 21.18008036337517; r 39.39179714868201
0.075: wreq 22.48436121832085; wresp 18.402744156414798; w 40.88710537473565; rreq 18.389337099675302; rresp 22.567467704593202; r 40.9568048042685
0.0875: wreq 34.83817693283922; wresp 18.60999858198613; w 53.44817551482535; rreq 18.616898531613124; rresp

# DOR 2 VC (packet_length = 4)

In [28]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 2

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM

PACKET_LEN = 4

BUF_DEPTH = 2

In [29]:
from multiprocessing import Pipe, Process
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400]
# pir_list = [0.075]
pir_list = [pir / PACKET_LEN for pir in pir_list]
calculation_tasks = list()
task_result_receivers = list()

for pir in pir_list:
  result_sender, result_receiver = Pipe()
  p = Process(target=calculation_task, args=(pir, result_sender))
  p.start()
  calculation_tasks.append(p)
  task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  input_lat_wreq.append(input_lat[0][0].sum() / PAIR_NUM + 0.5)
  output_lat_wreq.append(output_lat[0][0].sum() / PAIR_NUM + 0.5)
  lat_wreq.append(lat[0][0].sum() / PAIR_NUM + 1)
  input_lat_wresp.append(input_lat[1][1].sum() / PAIR_NUM + 0.5)
  output_lat_wresp.append(output_lat[1][1].sum() / PAIR_NUM + 0.5)
  lat_wresp.append(lat[1][1].sum() / PAIR_NUM + 1)
  input_lat_rreq.append(input_lat[1][0].sum() / PAIR_NUM + 0.5)
  output_lat_rreq.append(output_lat[1][0].sum() / PAIR_NUM + 0.5)
  lat_rreq.append(lat[1][0].sum() / PAIR_NUM + 1)
  input_lat_rresp.append(input_lat[0][1].sum() / PAIR_NUM + 0.5)
  output_lat_rresp.append(output_lat[0][1].sum() / PAIR_NUM + 0.5)
  lat_rresp.append(lat[0][1].sum() / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][0][0][11] + 1
  test_wresp = lat[1][1][11][0] + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")

pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=4; input_diff=5.783581753691855e-11; output_diff=2.4638569060996224e-11
0.00625 done
latency_wreq = 20.970149378800514; latency_wresp = 17.699335976457142; latency_write = 38.669485355257656
latency_rreq = 17.69813690444968; latency_rresp = 20.992464525997224; latency_read = 38.690601430446904
latency_write [0][0] -> [1][1]: wreq = 10.083001118418338; wresp = 7.002046247207299; write = 17.085047365625638
finished in iter=6; input_diff=2.1486412666276204e-11; output_diff=8.968932789400702e-12
0.0125 done
latency_wreq = 21.456749450602498; latency_wresp = 17.73689947892703; latency_write = 39.19364892952953
latency_rreq = 17.7348420949478; latency_rresp = 21.502324499986955; latency_read = 39.237166594934756
latency_write [0][0] -> [1][1]: wreq = 10.21632353273425; wresp = 7.004993784965455; write = 17.221317317699707
finished in iter=7; input_diff=2.7813579500943494e-10; output_diff=8.098570411312841e-11
0.01875 done
latency_wreq = 22.139545308882433; latency_wresp = 17

In [30]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("dor_results_p4.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [31]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.00625: wreq 20.970149378800514; wresp 17.699335976457142; w 38.669485355257656; rreq 17.69813690444968; rresp 20.992464525997224; r 38.690601430446904
0.0125: wreq 21.456749450602498; wresp 17.73689947892703; w 39.19364892952953; rreq 17.7348420949478; rresp 21.502324499986955; r 39.237166594934756
0.01875: wreq 22.139545308882433; wresp 17.77939672349531; w 39.91894203237774; rreq 17.776832694827174; rresp 22.209428587468906; r 39.986261282296084
0.025: wreq 23.050321122055355; wresp 17.826889281136506; w 40.87721040319186; rreq 17.824186929354816; rresp 23.145553228284516; r 40.969740157639336
0.03125: wreq 24.25881334891143; wresp 17.87946184630532; w 42.13827519521675; rreq 17.87701261155098; rresp 24.379866290288234; r 42.256878901839215
0.0375: wreq 25.955804895135; wresp 17.937223847376067; w 43.89302874251106; rreq 17.935450076426353; rresp 26.099421685848295; r 44.03487176227465
0.04375: wreq 29.519386187320922; wresp 18.0003116463226; w 47.51969783364352; rreq 17.9996771741

# DOR 2 VC (packet 1) + increased buffer

In [23]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 2

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM

PACKET_LEN = 1

BUF_DEPTH = 4

In [25]:
from multiprocessing import Pipe, Process
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400]
# pir_list = [0.075]
pir_list = [pir / PACKET_LEN for pir in pir_list]
# calculation_tasks = list()
# task_result_receivers = list()

# for pir in pir_list:
#   result_sender, result_receiver = Pipe()
#   p = Process(target=calculation_task, args=(pir, result_sender))
#   p.start()
#   calculation_tasks.append(p)
#   task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  # (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  (lat, input_lat, output_lat) = calc_average_latency_pir_pipeline(pir_list[i], max_iter=50)
  input_lat_wreq.append(input_lat[0][0].sum() / PAIR_NUM + 0.5)
  output_lat_wreq.append(output_lat[0][0].sum() / PAIR_NUM + 0.5)
  lat_wreq.append(lat[0][0].sum() / PAIR_NUM + 1)
  input_lat_wresp.append(input_lat[1][1].sum() / PAIR_NUM + 0.5)
  output_lat_wresp.append(output_lat[1][1].sum() / PAIR_NUM + 0.5)
  lat_wresp.append(lat[1][1].sum() / PAIR_NUM + 1)
  input_lat_rreq.append(input_lat[1][0].sum() / PAIR_NUM + 0.5)
  output_lat_rreq.append(output_lat[1][0].sum() / PAIR_NUM + 0.5)
  lat_rreq.append(lat[1][0].sum() / PAIR_NUM + 1)
  input_lat_rresp.append(input_lat[0][1].sum() / PAIR_NUM + 0.5)
  output_lat_rresp.append(output_lat[0][1].sum() / PAIR_NUM + 0.5)
  lat_rresp.append(lat[0][1].sum() / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][0][0][11] + 1
  test_wresp = lat[1][1][11][0] + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")

pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=2; input_diff=5.589182280224909e-13; output_diff=2.2929116853800144e-13
0.025 done
latency_wreq = 17.805317951366597; latency_wresp = 17.810846791000145; latency_write = 35.61616474236674
latency_rreq = 17.805317951366597; latency_rresp = 17.810846791000145; latency_read = 35.61616474236674
latency_write [0][0] -> [1][1]: wreq = 7.018823889953913; wresp = 7.01060897988798; write = 14.029432869841893
finished in iter=3; input_diff=2.0761788860163667e-12; output_diff=5.873510894915725e-13
0.05 done
latency_wreq = 17.998707312948227; latency_wresp = 18.00989527748594; latency_write = 36.008602590434165
latency_rreq = 17.998707312948227; latency_rresp = 18.00989527748594; latency_read = 36.008602590434165
latency_write [0][0] -> [1][1]: wreq = 7.047310555712658; wresp = 7.030254932076245; write = 14.077565487788902
finished in iter=4; input_diff=4.444551092251136e-12; output_diff=1.5629022416907539e-12
0.075 done
latency_wreq = 18.2634949003401; latency_wresp = 18.28049028

In [26]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("dor_results_2xbuf_depth.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [27]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.025: wreq 17.805317951366597; wresp 17.810846791000145; w 35.61616474236674; rreq 17.805317951366597; rresp 17.810846791000145; r 35.61616474236674
0.05: wreq 17.998707312948227; wresp 18.00989527748594; w 36.008602590434165; rreq 17.998707312948227; rresp 18.00989527748594; r 36.008602590434165
0.075: wreq 18.2634949003401; wresp 18.280490281856494; w 36.54398518219659; rreq 18.2634949003401; rresp 18.280490281856494; r 36.54398518219659
0.1: wreq 18.624350455082297; wresp 18.647322182841673; w 37.27167263792397; rreq 18.624350455082297; rresp 18.647322182841673; r 37.27167263792397
0.125: wreq 19.120805289401567; wresp 19.149916407119996; w 38.27072169652156; rreq 19.120805289401567; rresp 19.149916407119996; r 38.27072169652156
0.15: wreq 19.827115840432242; wresp 19.862316320096596; w 39.68943216052884; rreq 19.827115840432242; rresp 19.862316320096596; r 39.68943216052884
0.175: wreq 20.939340212337992; wresp 20.978822451145923; w 41.91816266348391; rreq 20.939340212337992; rres

# DOR 4 VC (packet 1)

In [32]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 4

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM
PACKET_LEN = 1

BUF_DEPTH = 2

In [33]:
from multiprocessing import Pipe, Process
# pir_list = [0.025, 0.05]
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450, 0.475, 0.500]
pir_list = [pir / PACKET_LEN for pir in pir_list]
calculation_tasks = list()
task_result_receivers = list()

for pir in pir_list:
  result_sender, result_receiver = Pipe()
  p = Process(target=calculation_task, args=(pir, result_sender))
  p.start()
  calculation_tasks.append(p)
  task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  input_lat_wreq.append((input_lat[0][0].sum() + input_lat[0][2].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_wreq.append((output_lat[0][0].sum() + output_lat[0][2].sum()) / 2 / PAIR_NUM + 0.5)
  lat_wreq.append((lat[0][0].sum() + lat[0][2].sum()) / 2 / PAIR_NUM + 1)
  input_lat_wresp.append((input_lat[1][1].sum() + input_lat[1][3].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_wresp.append((output_lat[1][1].sum() + output_lat[1][3].sum()) / 2 / PAIR_NUM + 0.5)
  lat_wresp.append((lat[1][1].sum() + lat[1][3].sum()) / 2 / PAIR_NUM + 1)
  input_lat_rreq.append((input_lat[1][0].sum() + input_lat[1][2].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_rreq.append((output_lat[1][0].sum() + output_lat[1][2].sum()) / 2 / PAIR_NUM + 0.5)
  lat_rreq.append((lat[1][0].sum() + lat[1][2].sum()) / 2 / PAIR_NUM + 1)
  input_lat_rresp.append((input_lat[0][1].sum() + input_lat[0][3].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_rresp.append((output_lat[0][1].sum() + output_lat[0][3].sum()) / 2 / PAIR_NUM + 0.5)
  lat_rresp.append((lat[0][1].sum() + lat[0][3].sum()) / 2 / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = (lat[0][2][0][11] + lat[0][0][0][11]) / 2 + 1
  test_wresp = (lat[1][3][11][0] + lat[1][1][11][0]) / 2 + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=3; input_diff=1.1538621115182958e-10; output_diff=3.7860823820101287e-11
0.025 done
latency_wreq = 17.84095710381434; latency_wresp = 17.84655627997824; latency_write = 35.68751338379258
latency_rreq = 17.84095710381434; latency_rresp = 17.84655627997824; latency_read = 35.68751338379258
latency_write [0][0] -> [1][1]: wreq = 7.027382754954259; wresp = 7.02561532092505; write = 14.052998075879309
finished in iter=4; input_diff=1.9744506552995018e-10; output_diff=7.520201812053674e-11
0.05 done
latency_wreq = 18.049694935708064; latency_wresp = 18.061176587325; latency_write = 36.11087152303307
latency_rreq = 18.049694935708064; latency_rresp = 18.061176587325; latency_read = 36.11087152303307
latency_write [0][0] -> [1][1]: wreq = 7.060153654556438; wresp = 7.055533676004867; write = 14.115687330561304
finished in iter=5; input_diff=2.3280237314663062e-10; output_diff=6.626811397233205e-11
0.075 done
latency_wreq = 18.30032970816981; latency_wresp = 18.318014227259226;

In [34]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("dor_4vc_results_p1.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [35]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.025: wreq 17.84095710381434; wresp 17.84655627997824; w 35.68751338379258; rreq 17.84095710381434; rresp 17.84655627997824; r 35.68751338379258
0.05: wreq 18.049694935708064; wresp 18.061176587325; w 36.11087152303307; rreq 18.049694935708064; rresp 18.061176587325; r 36.11087152303307
0.075: wreq 18.30032970816981; wresp 18.318014227259226; w 36.61834393542904; rreq 18.30032970816981; rresp 18.318014227259226; r 36.61834393542904
0.1: wreq 18.605069269591002; wresp 18.629316998229335; w 37.23438626782034; rreq 18.605069269591002; rresp 18.629316998229335; r 37.23438626782034
0.125: wreq 18.98610113021599; wresp 19.017305882546474; w 38.003407012762466; rreq 18.98610113021599; rresp 19.017305882546474; r 38.003407012762466
0.15: wreq 19.49201504242946; wresp 19.530541207000752; w 39.02255624943021; rreq 19.49201504242946; rresp 19.530541207000752; r 39.02255624943021
0.175: wreq 20.28573518700276; wresp 20.3312618778447; w 40.61699706484746; rreq 20.28573518700276; rresp 20.331261877

# DOR 4 VC (packet 2)

In [36]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 4

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM
PACKET_LEN = 2

BUF_DEPTH = 2

In [37]:
from multiprocessing import Pipe, Process
# pir_list = [0.025, 0.05]
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450, 0.475, 0.500]
pir_list = [pir / PACKET_LEN for pir in pir_list]
calculation_tasks = list()
task_result_receivers = list()

for pir in pir_list:
  result_sender, result_receiver = Pipe()
  p = Process(target=calculation_task, args=(pir, result_sender))
  p.start()
  calculation_tasks.append(p)
  task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  input_lat_wreq.append((input_lat[0][0].sum() + input_lat[0][2].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_wreq.append((output_lat[0][0].sum() + output_lat[0][2].sum()) / 2 / PAIR_NUM + 0.5)
  lat_wreq.append((lat[0][0].sum() + lat[0][2].sum()) / 2 / PAIR_NUM + 1)
  input_lat_wresp.append((input_lat[1][1].sum() + input_lat[1][3].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_wresp.append((output_lat[1][1].sum() + output_lat[1][3].sum()) / 2 / PAIR_NUM + 0.5)
  lat_wresp.append((lat[1][1].sum() + lat[1][3].sum()) / 2 / PAIR_NUM + 1)
  input_lat_rreq.append((input_lat[1][0].sum() + input_lat[1][2].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_rreq.append((output_lat[1][0].sum() + output_lat[1][2].sum()) / 2 / PAIR_NUM + 0.5)
  lat_rreq.append((lat[1][0].sum() + lat[1][2].sum()) / 2 / PAIR_NUM + 1)
  input_lat_rresp.append((input_lat[0][1].sum() + input_lat[0][3].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_rresp.append((output_lat[0][1].sum() + output_lat[0][3].sum()) / 2 / PAIR_NUM + 0.5)
  lat_rresp.append((lat[0][1].sum() + lat[0][3].sum()) / 2 / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = (lat[0][2][0][11] + lat[0][0][0][11]) / 2 + 1
  test_wresp = (lat[1][3][11][0] + lat[1][1][11][0]) / 2 + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=3; input_diff=8.13156458327136e-11; output_diff=2.658624597945939e-11
0.0125 done
latency_wreq = 18.86155405390791; latency_wresp = 17.752620190118858; latency_write = 36.61417424402677
latency_rreq = 17.749891152195627; latency_rresp = 18.869919536026284; latency_read = 36.61981068822191
latency_write [0][0] -> [1][1]: wreq = 8.035901322099093; wresp = 7.012275885613794; write = 15.048177207712886
finished in iter=4; input_diff=1.3776092260335735e-10; output_diff=5.236686328494626e-11
0.025 done
latency_wreq = 19.09988069049905; latency_wresp = 17.846556279986796; latency_write = 36.94643697048585
latency_rreq = 17.84110718198464; latency_rresp = 19.116967211856586; latency_read = 36.95807439384123
latency_write [0][0] -> [1][1]: wreq = 8.079924217936384; wresp = 7.025615320925118; write = 15.105539538861501
finished in iter=5; input_diff=1.6097787764997312e-10; output_diff=4.5293579935370277e-11
0.0375 done
latency_wreq = 19.389852231167815; latency_wresp = 17.949135

In [38]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("dor_4vc_results_p2.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [39]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.025: wreq 18.86155405390791; wresp 17.752620190118858; w 36.61417424402677; rreq 17.749891152195627; rresp 18.869919536026284; r 36.61981068822191
0.05: wreq 19.09988069049905; wresp 17.846556279986796; w 36.94643697048585; rreq 17.84110718198464; rresp 19.116967211856586; r 36.95807439384123
0.075: wreq 19.389852231167815; wresp 17.949135849045824; w 37.33898808021364; rreq 17.940973249663713; rresp 19.41606417264556; r 37.357037422309276
0.1: wreq 19.745045357309667; wresp 18.061176587346544; w 37.806221944656215; rreq 18.050304868012713; rresp 19.78083426108837; r 37.83113912910108
0.125: wreq 20.1902161506636; wresp 18.18370213613825; w 38.37391828680185; rreq 18.170123871398726; rresp 20.236059323992265; r 38.406183195390994
0.15: wreq 20.779488326277228; wresp 18.318014227393288; w 39.097502553670516; rreq 18.301730754223613; rresp 20.835772160836378; r 39.13750291505999
0.175: wreq 21.69296705300207; wresp 18.46580226380793; w 40.15876931681; rreq 18.44681458138855; rresp 21.7

# DOR 4 VC (packet 4)

In [40]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 4

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM
PACKET_LEN = 4

BUF_DEPTH = 2

In [41]:
from multiprocessing import Pipe, Process
# pir_list = [0.025, 0.05]
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450, 0.475, 0.500]
pir_list = [pir / PACKET_LEN for pir in pir_list]
calculation_tasks = list()
task_result_receivers = list()

for pir in pir_list:
  result_sender, result_receiver = Pipe()
  p = Process(target=calculation_task, args=(pir, result_sender))
  p.start()
  calculation_tasks.append(p)
  task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  input_lat_wreq.append((input_lat[0][0].sum() + input_lat[0][2].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_wreq.append((output_lat[0][0].sum() + output_lat[0][2].sum()) / 2 / PAIR_NUM + 0.5)
  lat_wreq.append((lat[0][0].sum() + lat[0][2].sum()) / 2 / PAIR_NUM + 1)
  input_lat_wresp.append((input_lat[1][1].sum() + input_lat[1][3].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_wresp.append((output_lat[1][1].sum() + output_lat[1][3].sum()) / 2 / PAIR_NUM + 0.5)
  lat_wresp.append((lat[1][1].sum() + lat[1][3].sum()) / 2 / PAIR_NUM + 1)
  input_lat_rreq.append((input_lat[1][0].sum() + input_lat[1][2].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_rreq.append((output_lat[1][0].sum() + output_lat[1][2].sum()) / 2 / PAIR_NUM + 0.5)
  lat_rreq.append((lat[1][0].sum() + lat[1][2].sum()) / 2 / PAIR_NUM + 1)
  input_lat_rresp.append((input_lat[0][1].sum() + input_lat[0][3].sum()) / 2 / PAIR_NUM + 0.5)
  output_lat_rresp.append((output_lat[0][1].sum() + output_lat[0][3].sum()) / 2 / PAIR_NUM + 0.5)
  lat_rresp.append((lat[0][1].sum() + lat[0][3].sum()) / 2 / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = (lat[0][2][0][11] + lat[0][0][0][11]) / 2 + 1
  test_wresp = (lat[1][3][11][0] + lat[1][1][11][0]) / 2 + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=3; input_diff=8.118147161590542e-11; output_diff=2.6495559753926303e-11
0.00625 done
latency_wreq = 20.902940220698465; latency_wresp = 17.70868180923263; latency_write = 38.6116220299311
latency_rreq = 17.707353222785212; latency_rresp = 20.91678379929391; latency_read = 38.624137022079125
latency_write [0][0] -> [1][1]: wreq = 10.05305805193936; wresp = 7.006005650465948; write = 17.059063702405307
finished in iter=4; input_diff=1.3687849093722354e-10; output_diff=5.19846959640901e-11
0.0125 done
latency_wreq = 21.201043888309016; latency_wresp = 17.75262019011889; latency_write = 38.953664078427906
latency_rreq = 17.750040662419863; latency_rresp = 21.229120116229712; latency_read = 38.97916077864957
latency_write [0][0] -> [1][1]: wreq = 10.119951616348713; wresp = 7.012275885613795; write = 17.132227501962507
finished in iter=5; input_diff=1.5930573230836888e-10; output_diff=4.457394071206197e-11
0.01875 done
latency_wreq = 21.570747872163945; latency_wresp = 17.7

In [42]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("dor_4vc_results_p4.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [43]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.025: wreq 20.902940220698465; wresp 17.70868180923263; w 38.6116220299311; rreq 17.707353222785212; rresp 20.91678379929391; r 38.624137022079125
0.05: wreq 21.201043888309016; wresp 17.75262019011889; w 38.953664078427906; rreq 17.750040662419863; rresp 21.229120116229712; r 38.97916077864957
0.075: wreq 21.570747872163945; wresp 17.79855232041644; w 39.36930019258038; rreq 17.794800467269326; rresp 21.613515171716294; r 39.40831563898562
0.1: wreq 22.02845973327155; wresp 17.846556279987; w 39.875016013258545; rreq 17.841712148298548; rresp 22.086436085178295; r 39.92814823347685
0.125: wreq 22.604258850941314; wresp 17.896718687428304; w 40.50097753836962; rreq 17.89086430317486; rresp 22.677974023664408; r 40.56883832683927
0.15: wreq 23.36381977387708; wresp 17.94913584904927; w 41.31295562292635; rreq 17.94235588821014; rresp 23.4536022033292; r 41.39595809153934
0.175: wreq 24.524051144234093; wresp 18.003915128110076; w 42.52796627234417; rreq 17.996297764015882; rresp 24.628

# MDOR (packet 1)

In [44]:
ROUTING = "MOD_DOR"
PHYS_NUM = 2
VIRT_NUM = 4

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM
PACKET_LEN = 1

BUF_DEPTH = 2

In [48]:
from multiprocessing import Pipe, Process
pir_list = [0.575, 0.6, 0.625, 0.650, 0.675, 0.700, 0.725, 0.750]
# pir_list = [1.0]
pir_list = [pir / PACKET_LEN for pir in pir_list]
calculation_tasks = list()
task_result_receivers = list()

for pir in pir_list:
  result_sender, result_receiver = Pipe()
  p = Process(target=calculation_task, args=(pir, result_sender))
  p.start()
  calculation_tasks.append(p)
  task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  input_lat_wreq.append((input_lat[0][0].sum() + input_lat[0][2].sum()) / PAIR_NUM + 0.5)
  output_lat_wreq.append((output_lat[0][0].sum() + output_lat[0][2].sum()) / PAIR_NUM + 0.5)
  lat_wreq.append((lat[0][0].sum() + lat[0][2].sum()) / PAIR_NUM + 1)
  input_lat_wresp.append((input_lat[1][1].sum() + input_lat[1][3].sum()) / PAIR_NUM + 0.5)
  output_lat_wresp.append((output_lat[1][1].sum() + output_lat[1][3].sum()) / PAIR_NUM + 0.5)
  lat_wresp.append((lat[1][1].sum() + lat[1][3].sum()) / PAIR_NUM + 1)
  input_lat_rreq.append((input_lat[1][0].sum() + input_lat[1][2].sum()) / PAIR_NUM + 0.5)
  output_lat_rreq.append((output_lat[1][0].sum() + output_lat[1][2].sum()) / PAIR_NUM + 0.5)
  lat_rreq.append((lat[1][0].sum() + lat[1][2].sum()) / PAIR_NUM + 1)
  input_lat_rresp.append((input_lat[0][1].sum() + input_lat[0][3].sum()) / PAIR_NUM + 0.5)
  output_lat_rresp.append((output_lat[0][1].sum() + output_lat[0][3].sum()) / PAIR_NUM + 0.5)
  lat_rresp.append((lat[0][1].sum() + lat[0][3].sum()) / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][2][0][11] + 1
  test_wresp = lat[1][3][11][0] + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=31; input_diff=3.955149146852022e-10; output_diff=2.078508751933104e-10
0.575 done
latency_wreq = 56.63265893516276; latency_wresp = 58.51631329435088; latency_write = 115.14897222951365
latency_rreq = 56.63265893516276; latency_rresp = 58.51631329435088; latency_read = 115.14897222951365
latency_write [0][0] -> [1][1]: wreq = 10.59722697711861; wresp = 9.946251556607578; write = 20.543478533726187
finished in iter=35; input_diff=8.924524585917113e-10; output_diff=5.160312218041076e-10
0.6 done
latency_wreq = 61.77411395632094; latency_wresp = 66.27208537576327; latency_write = 128.0461993320842
latency_rreq = 61.77411395632094; latency_rresp = 66.27208537576327; latency_read = 128.0461993320842
latency_write [0][0] -> [1][1]: wreq = 10.90701358581303; wresp = 10.331249913342326; write = 21.238263499155355
finished in iter=39; input_diff=7.034939674109308e-10; output_diff=3.7337550566546845e-10
0.625 done
latency_wreq = 68.76476504878704; latency_wresp = 76.73239975680

In [49]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("mod_dor_results_p1_vol2.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [50]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.575: wreq 56.63265893516276; wresp 58.51631329435088; w 115.14897222951365; rreq 56.63265893516276; rresp 58.51631329435088; r 115.14897222951365
0.6: wreq 61.77411395632094; wresp 66.27208537576327; w 128.0461993320842; rreq 61.77411395632094; rresp 66.27208537576327; r 128.0461993320842
0.625: wreq 68.76476504878704; wresp 76.73239975680433; w 145.49716480559135; rreq 68.76476504878704; rresp 76.73239975680433; r 145.49716480559135
0.65: wreq 76.52073507363993; wresp 92.11007832778856; w 168.6308134014285; rreq 76.52073507363993; rresp 92.11007832778856; r 168.6308134014285
0.675: wreq 87.030795850287; wresp 119.42588820461616; w 206.45668405490318; rreq 87.030795850287; rresp 119.42588820461616; r 206.45668405490318
0.7: wreq 113.072859661858; wresp 216.52027956721585; w 329.59313922907387; rreq 113.072859661858; rresp 216.52027956721585; r 329.59313922907387
0.725: wreq 327.5842387504878; wresp 515.5291438553922; w 843.1133826058801; rreq 327.5842387504878; rresp 515.529143855392

# MDOR (packet 2)

In [22]:
ROUTING = "MOD_DOR"
PHYS_NUM = 2
VIRT_NUM = 4

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM
PACKET_LEN = 2

BUF_DEPTH = 2

In [23]:
from multiprocessing import Pipe, Process
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450, 0.475, 0.500, 0.525, 0.55, 0.575, 0.6, 0.625, 0.650, 0.675, 0.700, 0.725, 0.750]
# pir_list = [1.0]
pir_list = [pir / PACKET_LEN for pir in pir_list]
calculation_tasks = list()
task_result_receivers = list()

# for pir in pir_list:
#   result_sender, result_receiver = Pipe()
#   p = Process(target=calculation_task, args=(pir, result_sender))
#   p.start()
#   calculation_tasks.append(p)
#   task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  # (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  (lat, input_lat, output_lat) = calc_average_latency_pir_pipeline(pir_list[i], max_iter=50)
  input_lat_wreq.append((input_lat[0][0].sum() + input_lat[0][2].sum()) / PAIR_NUM + 0.5)
  output_lat_wreq.append((output_lat[0][0].sum() + output_lat[0][2].sum()) / PAIR_NUM + 0.5)
  lat_wreq.append((lat[0][0].sum() + lat[0][2].sum()) / PAIR_NUM + 1)
  input_lat_wresp.append((input_lat[1][1].sum() + input_lat[1][3].sum()) / PAIR_NUM + 0.5)
  output_lat_wresp.append((output_lat[1][1].sum() + output_lat[1][3].sum()) / PAIR_NUM + 0.5)
  lat_wresp.append((lat[1][1].sum() + lat[1][3].sum()) / PAIR_NUM + 1)
  input_lat_rreq.append((input_lat[1][0].sum() + input_lat[1][2].sum()) / PAIR_NUM + 0.5)
  output_lat_rreq.append((output_lat[1][0].sum() + output_lat[1][2].sum()) / PAIR_NUM + 0.5)
  lat_rreq.append((lat[1][0].sum() + lat[1][2].sum()) / PAIR_NUM + 1)
  input_lat_rresp.append((input_lat[0][1].sum() + input_lat[0][3].sum()) / PAIR_NUM + 0.5)
  output_lat_rresp.append((output_lat[0][1].sum() + output_lat[0][3].sum()) / PAIR_NUM + 0.5)
  lat_rresp.append((lat[0][1].sum() + lat[0][3].sum()) / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][2][0][11] + 1
  test_wresp = lat[1][3][11][0] + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=3; input_diff=1.2215597469599661e-11; output_diff=5.787430955864356e-12
0.0125 done
latency_wreq = 18.806436103731794; latency_wresp = 17.730003132878764; latency_write = 36.53643923661056
latency_rreq = 17.727158927268714; latency_rresp = 18.821432352015805; latency_read = 36.54859127928452
latency_write [0][0] -> [1][1]: wreq = 8.02809795338552; wresp = 7.010445317500039; write = 15.03854327088556
finished in iter=4; input_diff=1.0931030740106151e-11; output_diff=2.7382071323381743e-12
0.025 done
latency_wreq = 18.979284587466807; latency_wresp = 17.799555044556552; latency_write = 36.77883963202336
latency_rreq = 17.793647850675175; latency_rresp = 19.01046746370669; latency_read = 36.80411531438186
latency_write [0][0] -> [1][1]: wreq = 8.068655983919015; wresp = 7.022827143544202; write = 15.091483127463217
finished in iter=4; input_diff=5.606177934217148e-10; output_diff=1.417783436748843e-10
0.0375 done
latency_wreq = 19.186173173955524; latency_wresp = 17.87541

In [24]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("mod_dor_results_p2.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [25]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.025: wreq 18.806436103731794; wresp 17.730003132878764; w 36.53643923661056; rreq 17.727158927268714; rresp 18.821432352015805; r 36.54859127928452
0.05: wreq 18.979284587466807; wresp 17.799555044556552; w 36.77883963202336; rreq 17.793647850675175; rresp 19.01046746370669; r 36.80411531438186
0.075: wreq 19.186173173955524; wresp 17.87541764685677; w 37.0615908208123; rreq 17.86622910238775; rresp 19.234778327931508; r 37.10100743031926
0.1: wreq 19.428640811719752; wresp 17.9577131722401; w 37.38635398395985; rreq 17.945024290016388; rresp 19.49597023351048; r 37.44099452352687
0.125: wreq 19.708886763983312; wresp 18.046592004737576; w 37.75547876872089; rreq 18.03018231172503; rresp 19.79632544836872; r 37.82650776009375
0.15: wreq 20.029902064088372; wresp 18.142234299849015; w 38.17213636393738; rreq 18.12188116826796; rresp 20.13893138575945; r 38.26081255402741
0.175: wreq 20.395669997206806; wresp 18.24485210495183; w 38.640522102158634; rreq 18.220330304928087; rresp 20.52

# MDOR (packet 4)

In [26]:
ROUTING = "MOD_DOR"
PHYS_NUM = 2
VIRT_NUM = 4

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM
PACKET_LEN = 4

BUF_DEPTH = 2

In [27]:
from multiprocessing import Pipe, Process
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450, 0.475, 0.500, 0.525, 0.55, 0.575, 0.6, 0.625, 0.650, 0.675, 0.700, 0.725, 0.750]
# pir_list = [1.0]
pir_list = [pir / PACKET_LEN for pir in pir_list]
calculation_tasks = list()
task_result_receivers = list()

# for pir in pir_list:
#   result_sender, result_receiver = Pipe()
#   p = Process(target=calculation_task, args=(pir, result_sender))
#   p.start()
#   calculation_tasks.append(p)
#   task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  # (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  (lat, input_lat, output_lat) = calc_average_latency_pir_pipeline(pir_list[i], max_iter=50)
  input_lat_wreq.append((input_lat[0][0].sum() + input_lat[0][2].sum()) / PAIR_NUM + 0.5)
  output_lat_wreq.append((output_lat[0][0].sum() + output_lat[0][2].sum()) / PAIR_NUM + 0.5)
  lat_wreq.append((lat[0][0].sum() + lat[0][2].sum()) / PAIR_NUM + 1)
  input_lat_wresp.append((input_lat[1][1].sum() + input_lat[1][3].sum()) / PAIR_NUM + 0.5)
  output_lat_wresp.append((output_lat[1][1].sum() + output_lat[1][3].sum()) / PAIR_NUM + 0.5)
  lat_wresp.append((lat[1][1].sum() + lat[1][3].sum()) / PAIR_NUM + 1)
  input_lat_rreq.append((input_lat[1][0].sum() + input_lat[1][2].sum()) / PAIR_NUM + 0.5)
  output_lat_rreq.append((output_lat[1][0].sum() + output_lat[1][2].sum()) / PAIR_NUM + 0.5)
  lat_rreq.append((lat[1][0].sum() + lat[1][2].sum()) / PAIR_NUM + 1)
  input_lat_rresp.append((input_lat[0][1].sum() + input_lat[0][3].sum()) / PAIR_NUM + 0.5)
  output_lat_rresp.append((output_lat[0][1].sum() + output_lat[0][3].sum()) / PAIR_NUM + 0.5)
  lat_rresp.append((lat[0][1].sum() + lat[0][3].sum()) / PAIR_NUM + 1)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][2][0][11] + 1
  test_wresp = lat[1][3][11][0] + 1
  test_write = test_wreq + test_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=3; input_diff=1.2209500094514614e-11; output_diff=5.782280561910474e-12
0.00625 done
latency_wreq = 20.832429506870035; latency_wresp = 17.697562890712; latency_write = 38.529992397582035
latency_rreq = 17.696206747091107; latency_rresp = 20.865299717686366; latency_read = 38.56150646477747
latency_write [0][0] -> [1][1]: wreq = 10.038940054592006; wresp = 7.004977990553236; write = 17.04391804514524
finished in iter=4; input_diff=1.0913900167008879e-11; output_diff=2.730556348220647e-12
0.0125 done
latency_wreq = 21.049494449680385; latency_wresp = 17.730003132878775; latency_write = 38.77949758255916
latency_rreq = 17.727313553659936; latency_rresp = 21.116438864886653; latency_read = 38.843752418546586
latency_write [0][0] -> [1][1]: wreq = 10.099581140342117; wresp = 7.010445317500039; write = 17.110026457842157
finished in iter=4; input_diff=5.59264973188507e-10; output_diff=1.4117727874283886e-10
0.01875 done
latency_wreq = 21.31862345047757; latency_wresp = 17.7

In [28]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("mod_dor_results_p4.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [29]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.025: wreq 20.832429506870035; wresp 17.697562890712; w 38.529992397582035; rreq 17.696206747091107; rresp 20.865299717686366; r 38.56150646477747
0.05: wreq 21.049494449680385; wresp 17.730003132878775; w 38.77949758255916; rreq 17.727313553659936; rresp 21.116438864886653; r 38.843752418546586
0.075: wreq 21.31862345047757; wresp 17.763996827052875; w 39.082620277530445; rreq 17.759997685262984; rresp 21.42094053951852; r 39.1809382247815
0.1: wreq 21.641402172234784; wresp 17.7995550445568; w 39.44095721679159; rreq 17.794271745193743; rresp 21.780507171524537; r 39.57477891671828
0.125: wreq 22.020331107538418; wresp 17.836690513870806; w 39.857021621409224; rreq 17.83015044917541; rresp 22.197767698016044; r 40.02791814719146
0.15: wreq 22.458981059867277; wresp 17.87541764686122; w 40.33439870672849; rreq 17.86765077318688; rresp 22.676423515924533; r 40.544074289111414
0.175: wreq 22.962236819616408; wresp 17.915752571644543; w 40.877989391260954; rreq 17.906792156423485; rresp

# 8+8 EU and 4+4 DMA | Mesh 1x10x6 DOR | 4-ch router

In [22]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 2

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 6
TILE_NUM = X_NUM * Y_NUM

PACKET_LEN = 1

BUF_DEPTH = 2

In [24]:
# from multiprocessing import Pipe, Process
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450]
# pir_list = [0.425, 0.450]
pir_list = [pir / PACKET_LEN for pir in pir_list]

lh_angle = Coord(0)
lh_angle.x = 0
lh_angle.y = 0
ll_angle = Coord(0)
ll_angle.x = 0
ll_angle.y = Y_NUM-1
rh_angle = Coord(0)
rh_angle.x = X_NUM-1
rh_angle.y = 0
rl_angle = Coord(0)
rl_angle.x = X_NUM-1
rl_angle.y = Y_NUM-1
s_exclude = list()
s_exclude.append(lh_angle.id())
s_exclude.append(ll_angle.id())
s_exclude.append(rh_angle.id())
s_exclude.append(rl_angle.id())
# calculation_tasks = list()
# task_result_receivers = list()

# for pir in pir_list:
#   result_sender, result_receiver = Pipe()
#   p = Process(target=calculation_task, args=(pir, result_sender))
#   p.start()
#   calculation_tasks.append(p)
#   task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2) - (X_NUM - 2) * (Y_NUM - 2) * len(s_exclude)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  # (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  (lat, input_lat, output_lat) = calc_average_latency_pir_pipeline(pir_list[i], s_exclude=s_exclude, max_iter=50)
  input_lat_wreq.append(input_lat[0][0].sum() / PAIR_NUM + 1)
  output_lat_wreq.append(output_lat[0][0].sum() / PAIR_NUM + 1)
  lat_wreq.append(lat[0][0].sum() / PAIR_NUM + 2)
  input_lat_wresp.append(input_lat[1][1].sum() / PAIR_NUM + 1)
  output_lat_wresp.append(output_lat[1][1].sum() / PAIR_NUM + 1)
  lat_wresp.append(lat[1][1].sum() / PAIR_NUM + 2)
  input_lat_rreq.append(input_lat[1][0].sum() / PAIR_NUM + 1)
  output_lat_rreq.append(output_lat[1][0].sum() / PAIR_NUM + 1)
  lat_rreq.append(lat[1][0].sum() / PAIR_NUM + 2)
  input_lat_rresp.append(input_lat[0][1].sum() / PAIR_NUM + 1)
  output_lat_rresp.append(output_lat[0][1].sum() / PAIR_NUM + 1)
  lat_rresp.append(lat[0][1].sum() / PAIR_NUM + 2)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][0][0][11]
  test_wresp = lat[1][1][11][0]
  test_write = test_wreq + test_wresp
  test2_wreq = lat[0][0][10][11]
  test2_wresp = lat[1][1][11][10]
  test2_write = test2_wreq + test2_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
  print(f"latency_write [0][1] -> [1][1]: wreq = {test2_wreq}; wresp = {test2_wresp}; write = {test2_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=3; input_diff=5.547242560013255e-10; output_diff=1.9676114194598798e-10
0.025 done
latency_wreq = 14.767973152775918; latency_wresp = 14.771032881473525; latency_write = 29.539006034249443
latency_rreq = 14.767973152775918; latency_rresp = 14.771032881473525; latency_read = 29.539006034249443
latency_write [0][0] -> [1][1]: wreq = 0.0; wresp = 0.0; write = 0.0
latency_write [0][1] -> [1][1]: wreq = 4.012787482672792; wresp = 4.008343034775356; write = 8.021130517448148
finished in iter=5; input_diff=1.0149239275255122e-11; output_diff=3.4664827169715804e-12
0.05 done
latency_wreq = 14.913092059893087; latency_wresp = 14.919048906415776; latency_write = 29.832140966308863
latency_rreq = 14.913092059893087; latency_rresp = 14.919048906415776; latency_read = 29.832140966308863
latency_write [0][0] -> [1][1]: wreq = 0.0; wresp = 0.0; write = 0.0
latency_write [0][1] -> [1][1]: wreq = 4.03481227578266; wresp = 4.023616938283818; write = 8.058429214066479
finished in iter=6;

In [25]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("1_10x6_dor_4ch.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [26]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.025: wreq 14.767973152775918; wresp 14.771032881473525; w 29.539006034249443; rreq 14.767973152775918; rresp 14.771032881473525; r 29.539006034249443
0.05: wreq 14.913092059893087; wresp 14.919048906415776; w 29.832140966308863; rreq 14.913092059893087; rresp 14.919048906415776; r 29.832140966308863
0.075: wreq 15.10426252979095; wresp 15.112879184233913; w 30.217141714024862; rreq 15.10426252979095; rresp 15.112879184233913; r 30.217141714024862
0.1: wreq 15.345544442566608; wresp 15.356459634546558; w 30.702004077113166; rreq 15.345544442566608; rresp 15.356459634546558; r 30.702004077113166
0.125: wreq 15.643390175315867; wresp 15.65604228877443; w 31.299432464090295; rreq 15.643390175315867; rresp 15.65604228877443; r 31.299432464090295
0.15: wreq 16.007741230412726; wresp 16.021241444856685; w 32.02898267526941; rreq 16.007741230412726; rresp 16.021241444856685; r 32.02898267526941
0.175: wreq 16.454185869662062; wresp 16.46708806673839; w 32.92127393640045; rreq 16.454185869662

# 8+8 EU and 4+4 DMA | Mesh 1x10x6 MOD_DOR | 4-ch router

In [27]:
ROUTING = "MOD_DOR"
PHYS_NUM = 2
VIRT_NUM = 4

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 6
TILE_NUM = X_NUM * Y_NUM

PACKET_LEN = 1

BUF_DEPTH = 2

In [28]:
# from multiprocessing import Pipe, Process
pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450, 0.475, 0.500, 0.525, 0.550]
# pir_list = [0.025, 0.500]
pir_list = [pir / PACKET_LEN for pir in pir_list]

lh_angle = Coord(0)
lh_angle.x = 0
lh_angle.y = 0
ll_angle = Coord(0)
ll_angle.x = 0
ll_angle.y = Y_NUM-1
rh_angle = Coord(0)
rh_angle.x = X_NUM-1
rh_angle.y = 0
rl_angle = Coord(0)
rl_angle.x = X_NUM-1
rl_angle.y = Y_NUM-1
s_exclude = list()
s_exclude.append(lh_angle.id())
s_exclude.append(ll_angle.id())
s_exclude.append(rh_angle.id())
s_exclude.append(rl_angle.id())
# calculation_tasks = list()
# task_result_receivers = list()

# for pir in pir_list:
#   result_sender, result_receiver = Pipe()
#   p = Process(target=calculation_task, args=(pir, result_sender))
#   p.start()
#   calculation_tasks.append(p)
#   task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2) - (X_NUM - 2) * (Y_NUM - 2) * len(s_exclude)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  # (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  (lat, input_lat, output_lat) = calc_average_latency_pir_pipeline(pir_list[i], s_exclude=s_exclude, max_iter=50)
  input_lat_wreq.append((input_lat[0][0].sum() + input_lat[0][2].sum()) / PAIR_NUM + 1)
  output_lat_wreq.append((output_lat[0][0].sum() + output_lat[0][2].sum()) / PAIR_NUM + 1)
  lat_wreq.append((lat[0][0].sum() + lat[0][2].sum()) / PAIR_NUM + 2)
  input_lat_wresp.append((input_lat[1][1].sum() + input_lat[1][3].sum()) / PAIR_NUM + 1)
  output_lat_wresp.append((output_lat[1][1].sum() + output_lat[1][3].sum()) / PAIR_NUM + 1)
  lat_wresp.append((lat[1][1].sum() + lat[1][3].sum()) / PAIR_NUM + 2)
  input_lat_rreq.append((input_lat[1][0].sum() + input_lat[1][2].sum()) / PAIR_NUM + 1)
  output_lat_rreq.append((output_lat[1][0].sum() + output_lat[1][2].sum()) / PAIR_NUM + 1)
  lat_rreq.append((lat[1][0].sum() + lat[1][2].sum()) / PAIR_NUM + 2)
  input_lat_rresp.append((input_lat[0][1].sum() + input_lat[0][3].sum()) / PAIR_NUM + 1)
  output_lat_rresp.append((output_lat[0][1].sum() + output_lat[0][3].sum()) / PAIR_NUM + 1)
  lat_rresp.append((lat[0][1].sum() + lat[0][3].sum()) / PAIR_NUM + 2)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][2][0][11]
  test_wresp = lat[1][3][11][0]
  test_write = test_wreq + test_wresp
  test2_wreq = lat[0][2][10][11]
  test2_wresp = lat[1][3][11][10]
  test2_write = test2_wreq + test2_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
  print(f"latency_write [0][1] -> [1][1]: wreq = {test2_wreq}; wresp = {test2_wresp}; write = {test2_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=3; input_diff=1.83656470904599e-11; output_diff=7.521775703970864e-12
0.025 done
latency_wreq = 14.777101217875957; latency_wresp = 14.780657886125454; latency_write = 29.557759104001413
latency_rreq = 14.777101217875957; latency_rresp = 14.780657886125454; latency_read = 29.557759104001413
latency_write [0][0] -> [1][1]: wreq = 0.0; wresp = 0.0; write = 0.0
latency_write [0][1] -> [1][1]: wreq = 4.010061317695501; wresp = 4.004866676242805; write = 8.014927993938306
finished in iter=4; input_diff=1.6325373260806016e-11; output_diff=5.6863283531622375e-12
0.05 done
latency_wreq = 14.912444867883275; latency_wresp = 14.920413678392956; latency_write = 29.83285854627623
latency_rreq = 14.912444867883275; latency_rresp = 14.920413678392956; latency_read = 29.83285854627623
latency_write [0][0] -> [1][1]: wreq = 0.0; wresp = 0.0; write = 0.0
latency_write [0][1] -> [1][1]: wreq = 4.027123786407842; wresp = 4.015880793440952; write = 8.043004579848795
finished in iter=4; in

In [29]:
# 141.0777045707261
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("2_10x6_mod_dor_4ch.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [30]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.025: wreq 14.777101217875957; wresp 14.780657886125454; w 29.557759104001413; rreq 14.777101217875957; rresp 14.780657886125454; r 29.557759104001413
0.05: wreq 14.912444867883275; wresp 14.920413678392956; w 29.83285854627623; rreq 14.912444867883275; rresp 14.920413678392956; r 29.83285854627623
0.075: wreq 15.074596902829512; wresp 15.087834143245823; w 30.162431046075334; rreq 15.074596902829512; rresp 15.087834143245823; r 30.162431046075334
0.1: wreq 15.266320912263717; wresp 15.28569701138158; w 30.5520179236453; rreq 15.266320912263717; rresp 15.28569701138158; r 30.5520179236453
0.125: wreq 15.491557276105103; wresp 15.517965745553495; w 31.009523021658598; rreq 15.491557276105103; rresp 15.517965745553495; r 31.009523021658598
0.15: wreq 15.755952087966804; wresp 15.790312461336208; w 31.54626454930301; rreq 15.755952087966804; rresp 15.790312461336208; r 31.54626454930301
0.175: wreq 16.06779571343398; wresp 16.111045440722343; w 32.17884115415632; rreq 16.06779571343398; 

# 8+8 EU and 4+4 DMA | Mesh 1x10x10 DOR | 4-ch router

In [31]:
ROUTING = "DOR"
PHYS_NUM = 2
VIRT_NUM = 2

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM

PACKET_LEN = 1

BUF_DEPTH = 2

In [ ]:
# from multiprocessing import Pipe, Process
# pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450, 0.475, 0.500, 0.525, 0.550, 0.575, 0.600]
pir_list = [0.625, 0.650, 0.675, 0.700]
pir_list = [pir / PACKET_LEN for pir in pir_list]

lh_angle = Coord(0)
lh_angle.x = 0
lh_angle.y = 0
lh_angle_1 = Coord(0)
lh_angle_1.x = 0
lh_angle_1.y = 1
lh_angle_2 = Coord(0)
lh_angle_2.x = 0
lh_angle_2.y = 2
ll_angle = Coord(0)
ll_angle.x = 0
ll_angle.y = Y_NUM-1
ll_angle_1 = Coord(0)
ll_angle_1.x = 0
ll_angle_1.y = Y_NUM-2
ll_angle_2 = Coord(0)
ll_angle_2.x = 0
ll_angle_2.y = Y_NUM-3
rh_angle = Coord(0)
rh_angle.x = X_NUM-1
rh_angle.y = 0
rh_angle_1 = Coord(0)
rh_angle_1.x = X_NUM-1
rh_angle_1.y = 1
rh_angle_2 = Coord(0)
rh_angle_2.x = X_NUM-1
rh_angle_2.y = 2
rl_angle = Coord(0)
rl_angle.x = X_NUM-1
rl_angle.y = Y_NUM-1
rl_angle_1 = Coord(0)
rl_angle_1.x = X_NUM-1
rl_angle_1.y = Y_NUM-2
rl_angle_2 = Coord(0)
rl_angle_2.x = X_NUM-1
rl_angle_2.y = Y_NUM-3
s_exclude = list()
s_exclude.append(lh_angle.id())
s_exclude.append(ll_angle.id())
s_exclude.append(rh_angle.id())
s_exclude.append(rl_angle.id())
s_exclude.append(lh_angle_1.id())
s_exclude.append(ll_angle_1.id())
s_exclude.append(rh_angle_1.id())
s_exclude.append(rl_angle_1.id())
s_exclude.append(lh_angle_2.id())
s_exclude.append(ll_angle_2.id())
s_exclude.append(rh_angle_2.id())
s_exclude.append(rl_angle_2.id())
# calculation_tasks = list()
# task_result_receivers = list()

# for pir in pir_list:
#   result_sender, result_receiver = Pipe()
#   p = Process(target=calculation_task, args=(pir, result_sender))
#   p.start()
#   calculation_tasks.append(p)
#   task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2) - (X_NUM - 2) * (Y_NUM - 2) * len(s_exclude)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  # (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  (lat, input_lat, output_lat) = calc_average_latency_pir_pipeline(pir_list[i], s_exclude=s_exclude, max_iter=50)
  input_lat_wreq.append(input_lat[0][0].sum() / PAIR_NUM + 1)
  output_lat_wreq.append(output_lat[0][0].sum() / PAIR_NUM + 1)
  lat_wreq.append(lat[0][0].sum() / PAIR_NUM + 2)
  input_lat_wresp.append(input_lat[1][1].sum() / PAIR_NUM + 1)
  output_lat_wresp.append(output_lat[1][1].sum() / PAIR_NUM + 1)
  lat_wresp.append(lat[1][1].sum() / PAIR_NUM + 2)
  input_lat_rreq.append(input_lat[1][0].sum() / PAIR_NUM + 1)
  output_lat_rreq.append(output_lat[1][0].sum() / PAIR_NUM + 1)
  lat_rreq.append(lat[1][0].sum() / PAIR_NUM + 2)
  input_lat_rresp.append(input_lat[0][1].sum() / PAIR_NUM + 1)
  output_lat_rresp.append(output_lat[0][1].sum() / PAIR_NUM + 1)
  lat_rresp.append(lat[0][1].sum() / PAIR_NUM + 2)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][0][0][11]
  test_wresp = lat[1][1][11][0]
  test_write = test_wreq + test_wresp
  test2_wreq = lat[0][0][10][11]
  test2_wresp = lat[1][1][11][10]
  test2_write = test2_wreq + test2_wresp
  test3_wreq = lat[0][0][30][11]
  test3_wresp = lat[1][1][11][30]
  test3_write = test2_wreq + test2_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
  print(f"latency_write [0][1] -> [1][1]: wreq = {test2_wreq}; wresp = {test2_wresp}; write = {test2_write}")
  print(f"latency_write [0][3] -> [1][1]: wreq = {test3_wreq}; wresp = {test3_wresp}; write = {test3_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

In [ ]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("3_10x10_dor_4ch_v3.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [37]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.525: wreq 27.751721982244614; wresp 28.935890259459285; w 56.6876122417039; rreq 27.751721982244614; rresp 28.935890259459285; r 56.6876122417039
0.55: wreq 29.764905805552853; wresp 31.543979923459684; w 61.30888572901254; rreq 29.764905805552853; rresp 31.543979923459684; r 61.30888572901254
0.575: wreq 32.523843298924234; wresp 34.23044482213271; w 66.75428812105694; rreq 32.523843298924234; rresp 34.23044482213271; r 66.75428812105694
0.6: wreq 35.32642142904793; wresp 36.88572947571131; w 72.21215090475924; rreq 35.32642142904793; rresp 36.88572947571131; r 72.21215090475924


# 8+8 EU and 4+4 DMA | Mesh 1x10x10 MOD_DOR | 4-ch router

In [22]:
ROUTING = "MOD_DOR"
PHYS_NUM = 2
VIRT_NUM = 4

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM

PACKET_LEN = 1

BUF_DEPTH = 2

In [26]:
# from multiprocessing import Pipe, Process
# pir_list = [0.025, 0.05, 0.075, 0.1, 0.125, 0.150, 0.175, 0.200, 0.225, 0.250, 0.275, 0.300, 0.325, 0.350, 0.375, 0.400, 0.425, 0.450, 0.475, 0.500, 0.525, 0.550, 0.575, 0.600, 0.625, 0.650]
pir_list = [0.625, 0.650, 0.675, 0.700, 0.725, 0.750, 0.775, 0.800]
pir_list = [pir / PACKET_LEN for pir in pir_list]

lh_angle = Coord(0)
lh_angle.x = 0
lh_angle.y = 0
lh_angle_1 = Coord(0)
lh_angle_1.x = 0
lh_angle_1.y = 1
lh_angle_2 = Coord(0)
lh_angle_2.x = 0
lh_angle_2.y = 2
ll_angle = Coord(0)
ll_angle.x = 0
ll_angle.y = Y_NUM-1
ll_angle_1 = Coord(0)
ll_angle_1.x = 0
ll_angle_1.y = Y_NUM-2
ll_angle_2 = Coord(0)
ll_angle_2.x = 0
ll_angle_2.y = Y_NUM-3
rh_angle = Coord(0)
rh_angle.x = X_NUM-1
rh_angle.y = 0
rh_angle_1 = Coord(0)
rh_angle_1.x = X_NUM-1
rh_angle_1.y = 1
rh_angle_2 = Coord(0)
rh_angle_2.x = X_NUM-1
rh_angle_2.y = 2
rl_angle = Coord(0)
rl_angle.x = X_NUM-1
rl_angle.y = Y_NUM-1
rl_angle_1 = Coord(0)
rl_angle_1.x = X_NUM-1
rl_angle_1.y = Y_NUM-2
rl_angle_2 = Coord(0)
rl_angle_2.x = X_NUM-1
rl_angle_2.y = Y_NUM-3
s_exclude = list()
s_exclude.append(lh_angle.id())
s_exclude.append(ll_angle.id())
s_exclude.append(rh_angle.id())
s_exclude.append(rl_angle.id())
s_exclude.append(lh_angle_1.id())
s_exclude.append(ll_angle_1.id())
s_exclude.append(rh_angle_1.id())
s_exclude.append(rl_angle_1.id())
s_exclude.append(lh_angle_2.id())
s_exclude.append(ll_angle_2.id())
s_exclude.append(rh_angle_2.id())
s_exclude.append(rl_angle_2.id())
# calculation_tasks = list()
# task_result_receivers = list()

# for pir in pir_list:
#   result_sender, result_receiver = Pipe()
#   p = Process(target=calculation_task, args=(pir, result_sender))
#   p.start()
#   calculation_tasks.append(p)
#   task_result_receivers.append(result_receiver)

PAIR_NUM = 2 * (X_NUM + Y_NUM - 2) * (X_NUM - 2) * (Y_NUM - 2) - (X_NUM - 2) * (Y_NUM - 2) * len(s_exclude)
lat_wreq = list()
input_lat_wreq = list()
output_lat_wreq = list()
lat_wresp = list()
input_lat_wresp = list()
output_lat_wresp = list()
lat_write = list()
lat_rreq = list()
input_lat_rreq = list()
output_lat_rreq = list()
lat_rresp = list()
input_lat_rresp = list()
output_lat_rresp = list()
lat_read = list()
for i in range(len(pir_list)):
  # (lat, input_lat, output_lat) = task_result_receivers[i].recv()
  (lat, input_lat, output_lat) = calc_average_latency_pir_pipeline(pir_list[i], s_exclude=s_exclude, max_iter=50)
  input_lat_wreq.append((input_lat[0][0].sum() + input_lat[0][2].sum()) / PAIR_NUM + 1)
  output_lat_wreq.append((output_lat[0][0].sum() + output_lat[0][2].sum()) / PAIR_NUM + 1)
  lat_wreq.append((lat[0][0].sum() + lat[0][2].sum()) / PAIR_NUM + 2)
  input_lat_wresp.append((input_lat[1][1].sum() + input_lat[1][3].sum()) / PAIR_NUM + 1)
  output_lat_wresp.append((output_lat[1][1].sum() + output_lat[1][3].sum()) / PAIR_NUM + 1)
  lat_wresp.append((lat[1][1].sum() + lat[1][3].sum()) / PAIR_NUM + 2)
  input_lat_rreq.append((input_lat[1][0].sum() + input_lat[1][2].sum()) / PAIR_NUM + 1)
  output_lat_rreq.append((output_lat[1][0].sum() + output_lat[1][2].sum()) / PAIR_NUM + 1)
  lat_rreq.append((lat[1][0].sum() + lat[1][2].sum()) / PAIR_NUM + 2)
  input_lat_rresp.append((input_lat[0][1].sum() + input_lat[0][3].sum()) / PAIR_NUM + 1)
  output_lat_rresp.append((output_lat[0][1].sum() + output_lat[0][3].sum()) / PAIR_NUM + 1)
  lat_rresp.append((lat[0][1].sum() + lat[0][3].sum()) / PAIR_NUM + 2)
  lat_write.append(lat_wreq[-1] + lat_wresp[-1])
  lat_read.append(lat_rreq[-1] + lat_rresp[-1])
  test_wreq = lat[0][2][0][11]
  test_wresp = lat[1][3][11][0]
  test_write = test_wreq + test_wresp
  test2_wreq = lat[0][2][10][11]
  test2_wresp = lat[1][3][11][10]
  test2_write = test2_wreq + test2_wresp
  test3_wreq = lat[0][2][30][11]
  test3_wresp = lat[1][3][11][30]
  test3_write = test2_wreq + test2_wresp
  print(f"{pir_list[i]} done")
  print(f"latency_wreq = {lat_wreq[-1]}; latency_wresp = {lat_wresp[-1]}; latency_write = {lat_write[-1]}")
  print(f"latency_rreq = {lat_rreq[-1]}; latency_rresp = {lat_rresp[-1]}; latency_read = {lat_read[-1]}")
  print(f"latency_write [0][0] -> [1][1]: wreq = {test_wreq}; wresp = {test_wresp}; write = {test_write}")
  print(f"latency_write [0][1] -> [1][1]: wreq = {test2_wreq}; wresp = {test2_wresp}; write = {test2_write}")
  print(f"latency_write [0][3] -> [1][1]: wreq = {test3_wreq}; wresp = {test3_wresp}; write = {test3_write}")
pir_list = [pir * PACKET_LEN for pir in pir_list]

finished in iter=27; input_diff=3.4313398966886163e-10; output_diff=1.3936858038581445e-10
0.625 done
latency_wreq = 41.05635823876914; latency_wresp = 43.22389518829766; latency_write = 84.28025342706681
latency_rreq = 41.05635823876914; latency_rresp = 43.22389518829766; latency_read = 84.28025342706681
latency_write [0][0] -> [1][1]: wreq = 0.0; wresp = 0.0; write = 0.0
latency_write [0][1] -> [1][1]: wreq = 0.0; wresp = 0.0; write = 0.0
latency_write [0][3] -> [1][1]: wreq = 13.825857504911955; wresp = 12.043133321403488; write = 0.0
finished in iter=29; input_diff=8.073067118499746e-10; output_diff=3.720358603699547e-10
0.65 done
latency_wreq = 44.096876711151054; latency_wresp = 47.66055345520598; latency_write = 91.75743016635704
latency_rreq = 44.096876711151054; latency_rresp = 47.66055345520598; latency_read = 91.75743016635704
latency_write [0][0] -> [1][1]: wreq = 0.0; wresp = 0.0; write = 0.0
latency_write [0][1] -> [1][1]: wreq = 0.0; wresp = 0.0; write = 0.0
latency_writ

In [27]:
lists = [
        pir_list,
        input_lat_wreq,
        output_lat_wreq,
        input_lat_wresp,
        output_lat_wresp,
        lat_wreq,
        lat_wresp,
        lat_write,
        input_lat_rreq,
        output_lat_rreq,
        input_lat_rresp,
        output_lat_rresp,
        lat_rreq,
        lat_rresp,
        lat_read
    ]
headers = [
        "pir",
        "input_lat_wreq",
        "output_lat_wreq",
        "input_lat_wresp",
        "output_lat_wresp",
        "lat_wreq",
        "lat_wresp",
        "lat_write",
        "input_lat_rreq",
        "output_lat_rreq",
        "input_lat_rresp",
        "output_lat_rresp",
        "lat_rreq",
        "lat_rresp",
        "lat_read"
    ]
with open("4_10x10_mod_dor_4ch_v2.csv", "w", newline="") as f:
        writer = csv.writer(f, delimiter=';', quotechar=';')

        # Write headers (optional)
        writer.writerow(headers)

        # For each row index up to max_len, build a row
        for i in range(len(pir_list)):
            row = []
            for lst in lists:
                row.append(lst[i])
            writer.writerow(row)

In [28]:
for i in range(len(pir_list)):
    print(f"{pir_list[i]}: wreq {lat_wreq[i]}; wresp {lat_wresp[i]}; w {lat_wreq[i] + lat_wresp[i]}; rreq {lat_rreq[i]}; rresp {lat_rresp[i]}; r {lat_rreq[i] + lat_rresp[i]}")

0.625: wreq 41.05635823876914; wresp 43.22389518829766; w 84.28025342706681; rreq 41.05635823876914; rresp 43.22389518829766; r 84.28025342706681
0.65: wreq 44.096876711151054; wresp 47.66055345520598; w 91.75743016635704; rreq 44.096876711151054; rresp 47.66055345520598; r 91.75743016635704
0.675: wreq 47.22176864308968; wresp 53.805233806195815; w 101.0270024492855; rreq 47.22176864308968; rresp 53.805233806195815; r 101.0270024492855
0.7: wreq 51.15776435706119; wresp 62.24090322843929; w 113.39866758550048; rreq 51.15776435706119; rresp 62.24090322843929; r 113.39866758550048
0.725: wreq 56.32850165853131; wresp 81.87987181400663; w 138.20837347253794; rreq 56.32850165853131; rresp 81.87987181400663; r 138.20837347253794
0.75: wreq 65.76613661561427; wresp 139.7409864676406; w 205.50712308325487; rreq 65.76613661561427; rresp 139.7409864676406; r 205.50712308325487
0.775: wreq 77.38451381460939; wresp 251.53430819245477; w 328.91882200706414; rreq 77.38451381460939; rresp 251.53430